# 86 · Genes outside one-to-one orthology along the elastic curve

Every cross-species analysis so far kept HCOP's reciprocal-best one-to-one pairs. Paralog families and genes without
an ortholog were outside it. Notebook 62 looked at them on the retired scFates coordinate; this notebook asks its
questions again on the package's elastic curve, adds the ones 62 could not ask, and takes its hypotheses from the
literature.

**Protocol.** `docs/results/pt-non-one-to-one.md` (frozen before any fit).

| Part | Question |
|---|---|
| W | Within-species curves on PT-identity units, tested against the typical gene's shape |
| H1–H3 | Is the zonation the 1:1 space misses enriched, where is it, and is it a male program? |
| H4–H6 | Do paralogs co-position or partition the PT, and does that follow sequence divergence or genomic distance? |
| H7–H8 | Family units across species with the 1:1 analysis's own test; the single-copy gene against its co-orthologs |
| H9–H10, H13 | Genes without an ortholog; agreement with microdissection; dominance inside families |
| H11 | AKI: genes outside the 1:1 pool after injury |
| H12 | Named claims from the literature |

**What bounds the reading.** Two male control mice, one male human donor (two sections), two male IR mice. Human
positions are the weakest measurement (issues 6, 21). Every test runs across genes, not animals; all are descriptive
of this cohort.

## 1 · Setup

In [ ]:
import argparse
import gzip
import hashlib
import json
import os
import re
import sys
import time
from pathlib import Path

for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, '1')
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PKG = results_root / 'package_pseudospace'
EXPORT = {'hvm': PKG / 'hvm' / 'pt_units.h5ad', 'aki': PKG / 'aki' / 'pt_units.h5ad'}
STAGE01 = PKG / 'runs' / 'hvm' / '01_units'
QC_AKI = PKG / 'runs' / 'aki' / '02_qc' / 'units_qc.h5ad'
upstream13 = results_root / 'minimal_pt_scfates'
T62 = results_root / 'pt_non_one_to_one_orthologs'               # read only: coordinate-free inventory columns
T71 = results_root / 'pt_pathways_package_pseudospace'           # read only: universe, caches, statistics
T72 = results_root / 'pt_package_coordinate_validation'          # read only: transitions
T76 = results_root / 'pt_package_aki'                            # read only: caches, statistics, segment cuts
EXT = data_root / 'external'
XIONG = {s: EXT / 'xiong2023_sex_bias' / f'PT-{s}_fWTvsmWT.csv' for s in ('S1', 'S2', 'S3')}
CHEN21 = EXT / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz'
CHEN23 = EXT / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz'
PARALOGS = {'mouse': EXT / 'ensembl_compara_paralogs' / 'Compara.98.protein_default.paralogs.mus_musculus.tsv.gz',
            'human': EXT / 'ensembl_compara_paralogs' / 'Compara.98.protein_default.paralogs.homo_sapiens.tsv.gz'}
PARALOG_SHA = {'mouse': '4dc176c799720b71a801ca682e1be9797e44f381c44b9790ed382755d71a7de4',
               'human': 'eeb440f61b115e805b0812519b95f226b85e39448ca60b9a4123f06969a440c0'}
COMPARA_HM = EXT / 'ensembl_compara_human_mouse' / 'Compara.98.protein_default.homologies.human_mouse.tsv.gz'
CDNA = {'mouse': [EXT / 'ensembl_r98_transcripts' / 'Mus_musculus.GRCm38.cdna.all.fa.gz',
                  EXT / 'ensembl_r98_transcripts' / 'Mus_musculus.GRCm38.ncrna.fa.gz'],
        'human': [EXT / 'ensembl_r110_human_transcripts' / 'Homo_sapiens.GRCh38.cdna.all.fa.gz',
                  EXT / 'ensembl_r110_human_transcripts' / 'Homo_sapiens.GRCh38.ncrna.fa.gz']}
output = results_root / 'pt_non_one_to_one_package'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures, cache):
    folder.mkdir(parents=True, exist_ok=True)

DS = 'human_vs_mouse'
MOUSE, HUMAN = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
SPECIES = {'mouse': MOUSE, 'human': HUMAN}
CONTROLS, IRS = ['Ctrl1A2', 'Ctrl1A4'], ['IR2A2', 'IR2A4']
REQUIRED = [EXPORT['hvm'], EXPORT['aki'], QC_AKI, PKG / 'aki' / 'all_units_positions.csv',
            *[STAGE01 / f'{s}_units.h5ad' for s in MOUSE + HUMAN], upstream13 / 'ortholog_map_used.csv',
            T62 / 'tables' / 'gene_inventory_mouse.csv', T62 / 'tables' / 'gene_inventory_human.csv',
            T72 / 'tables' / 'transitions.csv', T76 / 'tables' / 'universe.csv', T76 / 'tables' / 'gene_statistics.csv',
            T76 / 'tables' / 'segment_units.csv',
            *[T71 / 'tables' / f for f in ('inputs.csv', f'universe_{DS}.csv', f'gene_sets_{DS}.json', f'gene_statistics_{DS}.csv')],
            *XIONG.values(), CHEN21, CHEN23, *PARALOGS.values(), COMPARA_HM, *CDNA['mouse'], *CDNA['human']]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 62, 70–72 and 76 first; see data/README.md)')

NOTEBOOK_LOGIC_VERSION = '86.non_one_to_one.1'
FINGERPRINT = {'hvm': 'efafc9ac2e99', 'aki': 'bb4a80c1b484'}
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '24'))
SEED = 86
# Protocol constants
DETECTION = .02                         # PT-expressed: >= 2% of units in both specimens (62)
NON = ('1:many', 'many:1', 'many:many')
OR_MIN, ALPHA = 1.5, .05
UNIT_SEP = {'mouse': .5, 'human': .15}  # H4b: separated peaks (mouse units; human curve position)
N_DRAWS, MIN_FAMILIES = 1000, 8
TANDEM = 1e6
CALIBRATION_MAX = .075
EFFECT, RATIO = .25, .25                # H11b: 76's level and specificity rules
PT_ENRICH = 1.
BASIS_DF, DF1 = 6, 6
N_BINS = 40                              # H4c, as 85
SEGMENTS = ('S1', 'S2', 'S3')

MM = 1 / 25.4
INK, LIGHT, GREY = '#0b0b0b', '#d9d8d4', '#9a9893'
CLASS_COLOR = {'1:1': '#9a9893', 'non-1:1': '#C0392B', 'no ortholog': '#7D3C98'}
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'HUK1_COR1': '#D55E00', 'HUK1_MED1': '#E69F00'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 200)
GUARDS, PRIMARY = [], []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only); raises when its cache was purged."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


def primary(test, p, statistic, note=''):
    PRIMARY.append({'test': test, 'p': float(p), 'statistic': statistic, 'note': note})


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)

## 2 · Inputs and guards (not cached)

G1 `hvm_inputs` reproduces 71; G2 72's transitions; G3 the stage-01 native panels match the export unit by unit;
G4 62's inventories list the panels; G6 external sha256 (G5, the AKI reuse guard, is in section 13).

In [ ]:
import anndata as ad
from scipy import sparse

from pseudospace import non_one_to_one as no
from pseudospace import paralog_zonation as pz
from pseudospace import published_findings as pf
from pseudospace import species_calls as sc
from pseudospace import transition_sharpness as ts
from pseudospace.continuous_descriptors import logistic_transitions
from pseudospace.package_inputs import aki_inputs, hvm_inputs, read_pt_units
from pseudospace.pathway_pipelines import common_grid
from pseudospace.stage_cache import cached_payload, digest

t0 = time.perf_counter()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
assert len(MT) == 13, MT
UNITS = read_pt_units(EXPORT['hvm'], 'hvm')
PROV = UNITS.uns['provenance']
assert str(PROV['notebook_logic_version']).startswith('70.'), PROV['notebook_logic_version']
assert PROV['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT['hvm'], PROV['fingerprints']['pass1']
INP = hvm_inputs(UNITS, data_root, orthologs, exclude_genes=MT)
GENES71 = list(INP['genes'])
universe71 = pd.read_csv(T71 / 'tables' / f'universe_{DS}.csv')
assert GENES71 == universe71[universe71.eligible.astype(bool)].gene.astype(str).tolist(), 'G1: universe differs from 71'
assert INP['gene_sets'] == json.loads((T71 / 'tables' / f'gene_sets_{DS}.json').read_text()), 'G1: gene sets differ from 71'
assert int(pd.read_csv(T71 / 'tables' / 'inputs.csv', index_col=0).loc['units in the common support', 'Human vs mouse PT']) == len(INP['position'])
POSITION = np.asarray(INP['position'], float)
SPECIMEN = INP['specimen'].astype(str)
SEGMENT = np.asarray(INP['segment']).astype(str)
LIBRARY = np.asarray(INP['library'], float)
OBS = INP['obs']
IDENT = OBS['primary_identity'].astype(str).to_numpy()
COND71 = stored(T71, f'gene_model_{DS}_species')
STAT71 = pd.read_csv(T71 / 'tables' / f'gene_statistics_{DS}.csv', header=[0, 1], index_col=0)
np.testing.assert_allclose(COND71['stat|T_spatial'], STAT71['condition'].T_spatial.to_numpy(), rtol=1e-12, atol=0)
GRID, KNOTS = COND71['grid'], COND71['knots']
np.testing.assert_allclose(GRID, common_grid(POSITION, SPECIMEN, 101), rtol=0, atol=1e-12)
GUARDS.append(f'G1: {len(GENES71):,} genes and {len(POSITION):,} units equal 71; 71\'s cached condition model reproduces '
              f'its T_spatial; knots {np.round(KNOTS, 3).tolist()}')

TRANSITIONS = pd.read_csv(T72 / 'tables' / 'transitions.csv')
TRANSITIONS = TRANSITIONS[TRANSITIONS.dataset.eq('hvm')].reset_index(drop=True)
again_t = logistic_transitions(POSITION, SEGMENT, SPECIMEN).merge(TRANSITIONS, on=['specimen', 'transition'])
assert len(again_t) == 8 and np.allclose(again_t.position_x, again_t.position_y, atol=1e-6)
MOUSE_CUTS = sc.mean_transitions(TRANSITIONS, MOUSE)
GUARDS.append(f'G2: transitions reproduce 72\'s; mouse cuts {MOUSE_CUTS[0]:.3f}, {MOUSE_CUTS[1]:.3f}')


def to_mouse(position):
    return sc.to_segment_units(position, MOUSE_CUTS)


GRID_UNITS = {'mouse': to_mouse(GRID), 'human': GRID.copy()}

# G3: native panels, unit by unit
NATIVE, PANEL = {}, {}
for species, names in SPECIES.items():
    blocks = []
    for name in names:
        native = ad.read_h5ad(STAGE01 / f'{name}_units.h5ad')
        rows = np.flatnonzero(SPECIMEN == name)
        ids = OBS.polygon_id.astype(str).to_numpy()[rows]
        assert pd.Index(ids).isin(native.obs_names).all() and pd.Index(ids).is_unique, f'G3: {name} units missing'
        x = sparse.csr_matrix(native[list(ids)].X, dtype=float)
        export_x = sparse.csr_matrix(UNITS[OBS.index[rows]].X, dtype=float)
        symbols = UNITS.var_names if species == 'mouse' else pd.Index(UNITS.var.human_symbol.astype(str))
        cols = native.var_names.get_indexer(symbols)
        assert (cols >= 0).all(), f'G3: {name} lacks export genes'
        assert (x[:, cols] != export_x).nnz == 0, f'G3: {name} counts differ from the export'
        if species in PANEL:
            assert PANEL[species].equals(native.var_names)
        PANEL[species] = pd.Index(native.var_names.astype(str))
        blocks.append((rows, x))
    order = np.concatenate([r for r, _ in blocks])
    NATIVE[species] = {'rows': order, 'counts': sparse.vstack([x for _, x in blocks]).tocsr()}
GUARDS.append('G3: every export unit is in its stage-01 file; shared genes identical; panels mouse '
              f"{len(PANEL['mouse']):,}, human {len(PANEL['human']):,} genes")

INV_COLUMNS = ['gene', 'gene_id', 'component', 'n_human', 'n_mouse', 'ortholog_class', 'class_support2', 'class_support3',
               'class_support5', 'paper_partner', 'probe_status', 'offtarget_genes', 'probe_clean', 'probe_clean_strict', 'n_probes']
INV = {}
for species in SPECIES:
    inv = pd.read_csv(T62 / 'tables' / f'gene_inventory_{species}.csv', usecols=INV_COLUMNS)
    assert inv.gene.astype(str).tolist() == PANEL[species].tolist(), f'G4: {species} inventory differs from the panel'
    inv = inv.set_index('gene')
    inv['family_clean'] = no.family_clean(inv)
    inv['non'] = inv.class_support3.isin(NON)
    INV[species] = inv
GUARDS.append('G4: 62\'s inventories list the stage-01 panels in order (coordinate-free columns only)')

EXTERNAL_SHA = {str(p.relative_to(data_root)): sha256(p) for p in
                [*XIONG.values(), CHEN21, CHEN23, *PARALOGS.values(), COMPARA_HM]}
for species, path in PARALOGS.items():
    assert EXTERNAL_SHA[str(path.relative_to(data_root))] == PARALOG_SHA[species], f'G6: {species} paralog file changed'
GUARDS.append('G6: external sha256 recorded; the Ensembl paralog files match the 2026-10-13 download')
print('\n'.join('· ' + g for g in GUARDS))
print(f'Inputs {time.perf_counter() - t0:.0f} s')

## 3 · W · Within-species curves

U_PT (Proximal_Tubule identity) is primary; U_all and the native-panel library are H1's sensitivities. Each run is
cached.

In [ ]:
code_key = digest([NOTEBOOK_LOGIC_VERSION, Path(no.__file__).read_text()])
SECONDS = {}


def cached(stage, compute, *, params=None, inputs=None):
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs=inputs, code=code_key)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


U_PT = IDENT == 'Proximal_Tubule'
print({s: int((U_PT & (SPECIMEN == s)).sum()) for s in MOUSE + HUMAN}, 'PT-identity units')


def native_library(species):
    x = NATIVE[species]['counts']
    keep = ~PANEL[species].str.upper().str.startswith('MT-')
    return np.asarray(x[:, np.flatnonzero(keep)].sum(axis=1)).ravel()


def species_data(species, units='pt', library='shared'):
    """Rows (in the species' native order), their counts, library, position and specimens for one W run."""
    rows = NATIVE[species]['rows']
    keep = U_PT[rows] if units == 'pt' else np.ones(len(rows), bool)
    lib = LIBRARY[rows] if library == 'shared' else native_library(species)
    return {'rows': rows[keep], 'counts': NATIVE[species]['counts'][keep], 'library': lib[keep],
            'position': POSITION[rows][keep], 'specimen': SPECIMEN[rows][keep], 'segment': SEGMENT[rows][keep]}


def expressed(species, d):
    names = SPECIES[species]
    det = np.array([np.asarray((d['counts'][d['specimen'] == n] > 0).mean(axis=0)).ravel() for n in names])
    not_mt = ~PANEL[species].str.upper().str.startswith('MT-')
    return (det >= DETECTION).all(axis=0) & not_mt


def w_run(species, units='pt', library='shared', columns=None, shapes=None, tag=None):
    """One W run. ``columns`` = (units x k counts over the species' native rows, names) replaces the PT-expressed
    genes, for family sums read against the genes' typical ``shapes``."""
    d = species_data(species, units, library)
    if columns is None:
        genes = np.flatnonzero(expressed(species, d))
        x = d['counts'][:, genes]
        names = list(PANEL[species][genes])
    else:
        rows = NATIVE[species]['rows']
        keep = U_PT[rows] if units == 'pt' else np.ones(len(rows), bool)
        x, names = sparse.csr_matrix(columns[0])[keep], list(columns[1])
    key = {'counts': digest(x), 'library': d['library'], 'position': d['position'], 'specimen': d['specimen']}
    stage = f"w_{tag or f'{species}_{units}_{library}'}"

    def compute():
        z = no.species_zonation(x, d['library'], d['position'], d['specimen'], KNOTS, GRID, shapes=shapes, n_jobs=N_JOBS)
        out = {'alpha': z['alpha'], 'alpha_converged': z['alpha_converged']}
        for k in ('shape', 'phi', 'centred', 'lr', 'p', 'q', 'converged', 'separated', 'detection'):
            for n, v in z[k].items():
                out[f'{k}|{n}'] = v
        return out
    payload = cached(stage, compute, params={'knots': KNOTS, 'grid': GRID, 'detection': DETECTION,
                                             'shapes': None if shapes is None else np.vstack(list(shapes.values()))},
                     inputs=key)
    z = {'alpha': payload['alpha'], 'alpha_converged': payload['alpha_converged'].astype(bool)}
    for k in ('shape', 'phi', 'centred', 'lr', 'p', 'q', 'converged', 'separated', 'detection'):
        z[k] = {n: payload[f'{k}|{n}'] for n in SPECIES[species]}
        if k in ('converged', 'separated'):
            z[k] = {n: v.astype(bool) for n, v in z[k].items()}
    calls = no.zonation_calls(z, names, GRID_UNITS[species])
    return {'data': d, 'genes': names, 'z': z, 'calls': calls}


W = {species: w_run(species) for species in SPECIES}
for species, w in W.items():
    c = w['calls']
    print(f"{species}: {len(c):,} PT-expressed genes; zonated {int(c.zonated.sum()):,}, strong {int(c.strong.sum()):,}")


def annotate(species, calls, d):
    """Class, probe and expression columns for a W calls table."""
    inv = INV[species].reindex(calls.index)
    out = calls.join(inv[['class_support2', 'class_support3', 'class_support5', 'component', 'probe_clean',
                          'probe_clean_strict', 'family_clean', 'non', 'paper_partner', 'n_probes']])
    idx = PANEL[species].get_indexer(calls.index)
    cp = []
    for n in SPECIES[species]:
        m = d['specimen'] == n
        cp.append(np.asarray(d['counts'][m][:, idx].sum(axis=0)).ravel() / d['library'][m].sum() * 1e4)
    out['log2_cp10k'] = np.log2(np.mean(cp, axis=0))
    shared = set(UNITS.var_names) if species == 'mouse' else set(UNITS.var.human_symbol.astype(str))
    out['in_shared_space'] = out.index.isin(shared)
    out['cls'] = np.where(out.class_support3.eq('1:1'), '1:1', np.where(out.non, 'non-1:1', out.class_support3))
    return out


CALLS = {s: annotate(s, W[s]['calls'], W[s]['data']) for s in SPECIES}
CALLS['mouse']['peak_segment'] = pd.cut(CALLS['mouse'].peak, [-np.inf, 1, 2, np.inf], right=False, labels=SEGMENTS).astype(str)

# W5: the label view on U_PT
from pseudospace.program_localization import segment_pseudobulk


def label_view(species, d, genes):
    idx = PANEL[species].get_indexer(genes)
    rates = segment_pseudobulk(d['counts'][:, idx], d['library'], d['specimen'], d['segment'],
                               {s: [f'PT-{s}'] for s in SEGMENTS}, genes=genes)
    per = {}
    for n in SPECIES[species]:
        r = pd.DataFrame({s: rates[s][n] for s in SEGMENTS})
        c = r.sub(r.mean(axis=1), axis=0)
        per[n] = c - c.median()
    mean = (per[SPECIES[species][0]] + per[SPECIES[species][1]]) / 2
    out = pd.DataFrame({'label_peak': mean.idxmax(axis=1)})
    for n in SPECIES[species]:
        out[f'label_peak_{n}'] = per[n].idxmax(axis=1)
    return out.join(mean.add_prefix('label_'))


for s in SPECIES:
    CALLS[s] = CALLS[s].join(label_view(s, W[s]['data'], list(CALLS[s].index)))
    CALLS[s].to_csv(tables / f'w_calls_{s}.csv')
display(pd.concat({s: pd.crosstab(CALLS[s].cls, [CALLS[s].probe_clean, CALLS[s].strong]) for s in SPECIES}))

## 4 · H1 · Is mouse PT zonation enriched for genes outside 1:1 orthology?

In [ ]:
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu, norm, spearmanr, wilcoxon, binomtest


def mh_test(calls, outcome, *, class_col='class_support3', probe='probe_clean', two_sided=False):
    t = calls[calls[probe].astype(bool) & calls[class_col].isin(('1:1',) + NON)]
    exposed = t[class_col].isin(NON).to_numpy()
    strata = pd.qcut(t.log2_cp10k.rank(method='first'), 10, labels=False).to_numpy()
    r = pf.mantel_haenszel(exposed, t[outcome].astype(bool).to_numpy(), strata)
    p = 2 * norm.sf(abs(r['z'])) if two_sided else r['p']
    share_non = t.loc[exposed, outcome].mean()
    share_one = t.loc[~exposed, outcome].mean()
    return {'OR_MH': r['or_mh'], 'z': r['z'], 'p': p, 'non-1:1 n': int(exposed.sum()), 'non-1:1 share': share_non,
            '1:1 n': int((~exposed).sum()), '1:1 share': share_one, 'non-1:1 called': int(t.loc[exposed, outcome].sum())}


H1_ROWS = []
for s in SPECIES:
    H1_ROWS.append({'species': s, 'setting': 'primary (strong)', **mh_test(CALLS[s], 'strong', two_sided=s == 'human')})
    H1_ROWS.append({'species': s, 'setting': 'zonated', **mh_test(CALLS[s], 'zonated', two_sided=s == 'human')})
    for sup in ('class_support2', 'class_support5'):
        H1_ROWS.append({'species': s, 'setting': f'strong, {sup[-8:]}', **mh_test(CALLS[s], 'strong', class_col=sup, two_sided=s == 'human')})
    H1_ROWS.append({'species': s, 'setting': 'strong, strict probes', **mh_test(CALLS[s], 'strong', probe='probe_clean_strict', two_sided=s == 'human')})
for s in SPECIES:
    for units, library in (('all', 'shared'), ('pt', 'native')):
        w = w_run(s, units, library)
        c = annotate(s, w['calls'], w['data'])
        H1_ROWS.append({'species': s, 'setting': f'strong, units {units}, library {library}',
                        **mh_test(c, 'strong', two_sided=s == 'human')})
H1 = pd.DataFrame(H1_ROWS)
display(H1.round(4))
H1.to_csv(tables / 'h1_enrichment.csv', index=False)
h1 = H1.iloc[0]
primary('H1 mouse: non-1:1 enriched among strongly zonated genes (MH OR, one-sided)', h1.p, f"OR {h1.OR_MH:.2f}")
H1_HOLDS = bool(h1.OR_MH >= OR_MIN and h1.p < ALPHA)
by_class = {}
for s in SPECIES:
    t = CALLS[s][CALLS[s].probe_clean.astype(bool)]
    by_class[s] = t.groupby('class_support3').agg(expressed=('strong', 'size'), zonated=('zonated', 'sum'), strong=('strong', 'sum'))
    by_class[s]['strong share'] = by_class[s].strong / by_class[s].expressed
H1_CLASSES = pd.concat(by_class, names=['species'])
display(H1_CLASSES.round(3))
H1_CLASSES.to_csv(tables / 'h1_by_class.csv')
hn = CALLS['human'][CALLS['human'].non & CALLS['human'].probe_clean.astype(bool)]
H1_PROBES = hn.groupby(pd.cut(hn.n_probes, [0, 1, 2, np.inf], labels=['1', '2', '≥ 3'])).agg(
    genes=('zonated', 'size'), zonated=('zonated', 'mean'), strong=('strong', 'mean'))
print('Human non-1:1 genes by probe count:')
display(H1_PROBES.round(3))
H1_PROBES.to_csv(tables / 'h1_human_by_probe_count.csv')
strong_non = CALLS['mouse'][CALLS['mouse'].strong & CALLS['mouse'].non & CALLS['mouse'].probe_clean.astype(bool)]
print(f"H1 holds: {H1_HOLDS}. Mouse strong non-1:1 genes: {len(strong_non)}, {int((~strong_non.in_shared_space).sum())} outside the shared space")

### H1b · Are the human counterparts of that zonation flat?

In [ ]:
HC = CALLS['human']
human_by_comp = INV['human'].reset_index().groupby('component').gene.apply(list)


def human_status(genes, min_probes=1):
    genes = [g for g in genes if g in PANEL['human']]
    if not genes:
        return 'no member on the panel'
    measured = [g for g in genes if g in HC.index and HC.loc[g, 'probe_clean'] and HC.loc[g, 'n_probes'] >= min_probes]
    if any(HC.loc[g, 'zonated'] for g in measured):
        return 'zonated'
    if measured:
        return 'PT-expressed but flat'
    return 'not PT-expressed'


m_ = CALLS['mouse']
sm_ = m_[m_.strong & m_.probe_clean.astype(bool) & m_.cls.isin(['1:1', 'non-1:1'])]
status = {}
for g, r in sm_.iterrows():
    if r.cls == '1:1':
        partners = INV['human'].index[INV['human'].component.eq(r.component)].tolist() if r.component >= 0 else []
    else:
        partners = human_by_comp.get(r.component, [])
    status[g] = human_status(partners)
sm_ = sm_.assign(human_status=pd.Series(status))
H1B = pd.crosstab(sm_.cls, sm_.human_status)
display(H1B)
tab1b = np.array([[int((sm_.cls.eq('non-1:1') & sm_.human_status.eq('zonated')).sum()), int((sm_.cls.eq('non-1:1') & sm_.human_status.ne('zonated')).sum())],
                  [int((sm_.cls.eq('1:1') & sm_.human_status.eq('zonated')).sum()), int((sm_.cls.eq('1:1') & sm_.human_status.ne('zonated')).sum())]])
h1b = fisher_exact(tab1b, alternative='less')
print(f'H1b: human counterpart zonated: non-1:1 {tab1b[0, 0]}/{tab1b[0].sum()}, 1:1 {tab1b[1, 0]}/{tab1b[1].sum()}; '
      f'OR {h1b.statistic:.2f}, one-sided p {h1b.pvalue:.3g}')
primary('H1b: human counterpart zonated less often for mouse non-1:1 than 1:1 (Fisher)', h1b.pvalue, f'OR {h1b.statistic:.2f}')
status2 = {}
for g, r in sm_.iterrows():
    partners = (INV['human'].index[INV['human'].component.eq(r.component)].tolist() if r.cls == '1:1' and r.component >= 0
                else human_by_comp.get(r.component, []))
    status2[g] = human_status(partners, min_probes=2)
z2 = pd.Series(status2).eq('zonated')
t2 = np.array([[int((sm_.cls.eq('non-1:1') & z2).sum()), int((sm_.cls.eq('non-1:1') & ~z2).sum())],
               [int((sm_.cls.eq('1:1') & z2).sum()), int((sm_.cls.eq('1:1') & ~z2).sum())]])
h1b2 = fisher_exact(t2, alternative='less')
print(f'H1b, human members with ≥ 2 probes: non-1:1 {t2[0, 0]}/{t2[0].sum()}, 1:1 {t2[1, 0]}/{t2[1].sum()}; '
      f'OR {h1b2.statistic:.2f}, one-sided p {h1b2.pvalue:.3g}')
H1B.to_csv(tables / 'h1b_human_counterparts.csv')
sm_[['cls', 'Z', 'peak', 'human_status']].to_csv(tables / 'h1b_genes.csv')

## 5 · H2 · Where the non-1:1 zonation sits (mouse)

In [ ]:
m = CALLS['mouse']
sm = m[m.strong & m.probe_clean.astype(bool) & m.cls.isin(['1:1', 'non-1:1'])]
H2_TABLE = pd.crosstab(sm.cls, sm.peak_segment).reindex(columns=list(SEGMENTS), fill_value=0)
chi = chi2_contingency(H2_TABLE.to_numpy())
mw = mannwhitneyu(sm.peak[sm.cls.eq('non-1:1')], sm.peak[sm.cls.eq('1:1')], alternative='two-sided')
display(H2_TABLE.assign(**{f'{s} share': H2_TABLE[s] / H2_TABLE.sum(axis=1) for s in SEGMENTS}).round(3))
print(f'H2 chi2 {chi.statistic:.2f} (p {chi.pvalue:.3g}); Mann–Whitney of peak position p {mw.pvalue:.3g}; '
      f"median peak non-1:1 {sm.peak[sm.cls.eq('non-1:1')].median():.2f}, 1:1 {sm.peak[sm.cls.eq('1:1')].median():.2f}")
primary('H2 mouse: peak segment by class (chi-square)', chi.pvalue, f'chi2 {chi.statistic:.2f}')
H2_LABEL = pd.crosstab(sm.cls, sm.label_peak).reindex(columns=list(SEGMENTS), fill_value=0)
display(H2_LABEL)

## 6 · H3 · Sex: male bias, and positions that move with sex

In [ ]:
xi = {}
for seg, path in XIONG.items():
    t = pd.read_csv(path)
    t = t[t.tested.astype(str).str.upper().eq('TRUE')] if 'tested' in t else t
    xi[seg] = t.drop_duplicates('symbol').set_index('symbol')[['log2FC', 'p_adj']]
male = pd.concat({s: (v.p_adj < .05) & (v.log2FC < -1) for s, v in xi.items()}, axis=1).fillna(False).any(axis=1)
female = pd.concat({s: (v.p_adj < .05) & (v.log2FC > 1) for s, v in xi.items()}, axis=1).fillna(False).any(axis=1)
m = CALLS['mouse']
m['xiong_measured'] = m.index.isin(male.index)
m['male_biased'] = male.reindex(m.index).fillna(False).astype(bool)
m['female_biased'] = female.reindex(m.index).fillna(False).astype(bool)
sm = m[m.strong & m.probe_clean.astype(bool) & m.cls.isin(['1:1', 'non-1:1']) & m.xiong_measured]
tab = pd.crosstab(sm.cls, sm.male_biased).reindex(index=['non-1:1', '1:1'], columns=[True, False], fill_value=0)
h3a = fisher_exact(tab.to_numpy(), alternative='greater')
print(f"H3a: male-biased among strong genes: non-1:1 {tab.loc['non-1:1', True]}/{tab.loc['non-1:1'].sum()}, "
      f"1:1 {tab.loc['1:1', True]}/{tab.loc['1:1'].sum()}; OR {h3a.statistic:.2f}, one-sided p {h3a.pvalue:.3g}; "
      f"Xiong covers {int(m.xiong_measured.sum()):,} of {len(m):,} PT-expressed genes")
primary('H3a mouse: male-biased share, non-1:1 > 1:1 among strong (Fisher)', h3a.pvalue, f'OR {h3a.statistic:.2f}')

c23 = pd.read_csv(CHEN23, sep='\t').dropna(subset=['Gene_symbol'])
c23 = c23.assign(_m=c23[['PTS1', 'PTS2', 'PTS3']].max(axis=1)).sort_values('_m', ascending=False).drop_duplicates('Gene_symbol').set_index('Gene_symbol')
male_peak = c23[['PTS1', 'PTS2', 'PTS3']].idxmax(axis=1).str.replace('PT', '')
female_peak = c23[['fPTS1', 'fPTS2', 'fPTS3']].idxmax(axis=1).str.replace('fPT', '')
ok23 = (c23[['PTS1', 'PTS2', 'PTS3']].max(axis=1) >= 1) & (c23[['fPTS1', 'fPTS2', 'fPTS3']].max(axis=1) >= 1)
m['chen23_male_peak'] = male_peak.reindex(m.index)
m['chen23_female_peak'] = female_peak.reindex(m.index)
m['chen23_ok'] = ok23.reindex(m.index).fillna(False).astype(bool)
sm = m[m.strong & m.probe_clean.astype(bool) & m.cls.isin(['1:1', 'non-1:1']) & m.chen23_ok]
moved = sm.chen23_male_peak.ne(sm.chen23_female_peak)
tab3b = pd.crosstab(sm.cls, moved).reindex(index=['non-1:1', '1:1'], columns=[True, False], fill_value=0)
h3b = fisher_exact(tab3b.to_numpy(), alternative='greater')
print(f"H3b: microdissection peak moves with sex: non-1:1 {tab3b.loc['non-1:1', True]}/{tab3b.loc['non-1:1'].sum()}, "
      f"1:1 {tab3b.loc['1:1', True]}/{tab3b.loc['1:1'].sum()}; OR {h3b.statistic:.2f}, one-sided p {h3b.pvalue:.3g}")
primary('H3b mouse: sex moves the microdissection peak more often for non-1:1 (Fisher)', h3b.pvalue, f'OR {h3b.statistic:.2f}')
H3_MOVED = sm[moved][['cls', 'Z', 'peak', 'peak_segment', 'chen23_male_peak', 'chen23_female_peak', 'male_biased', 'female_biased']]
display(H3_MOVED[H3_MOVED.cls.eq('non-1:1')].sort_values('Z', ascending=False))
zm = m[m.zonated & m.probe_clean.astype(bool) & m.xiong_measured]
H3C = pd.DataFrame([{'group': g, 'n': int(mask.sum()), 'median peak': zm.peak[mask].median(),
                     'p vs unbiased (MW, two-sided)': mannwhitneyu(zm.peak[mask], zm.peak[~zm.male_biased & ~zm.female_biased]).pvalue}
                    for g, mask in (('male-biased', zm.male_biased), ('female-biased', zm.female_biased))])
H3C = pd.concat([H3C, pd.DataFrame([{'group': 'unbiased', 'n': int((~zm.male_biased & ~zm.female_biased).sum()),
                                     'median peak': zm.peak[~zm.male_biased & ~zm.female_biased].median()}])], ignore_index=True)
display(H3C.round(3))
allx = m[m.probe_clean.astype(bool) & m.xiong_measured & m.cls.isin(['1:1', 'non-1:1'])].copy()
allx['sex_biased'] = allx.male_biased | allx.female_biased
H3D_ROWS = []
for outcome in ('sex_biased', 'male_biased', 'female_biased'):
    r = mh_test(allx.assign(probe_clean=True), outcome, two_sided=True)
    H3D_ROWS.append({'outcome': outcome, **r})
H3D = pd.DataFrame(H3D_ROWS)
display(H3D.round(4))
primary('H3d mouse: sex-biased share, non-1:1 vs 1:1 over PT-expressed genes (MH, two-sided)', H3D.iloc[0].p,
        f"OR {H3D.iloc[0].OR_MH:.2f}")
H3D.to_csv(tables / 'h3d_sex_bias_all_expressed.csv', index=False)
pd.concat({'H3a': tab, 'H3b': tab3b}).to_csv(tables / 'h3_tables.csv')
H3_MOVED.to_csv(tables / 'h3b_moved_genes.csv')
H3C.to_csv(tables / 'h3c_peaks_by_sex_bias.csv', index=False)

## 7 · H4 · Do paralogs co-position or partition the PT?

Pairs of zonated probe-clean paralogs in one orthogroup and species, against 1,000 matched random pairs each (62's
rule). H4b asks whether label means already separate the pairs that separate; H4c fits the handover (mouse).

In [ ]:
def mean_curves(species, calls_index):
    """Specimens' mean centred curve on the grid points both cover; also the per-specimen curves and that mask."""
    z = W[species]['z']
    idx = pd.Index(W[species]['genes']).get_indexer(calls_index)
    a, b = (z['centred'][n][idx] for n in SPECIES[species])
    both = np.isfinite(z['centred'][SPECIES[species][0]]).all(0) & np.isfinite(z['centred'][SPECIES[species][1]]).all(0)
    return (a[:, both] + b[:, both]) / 2, {n: z['centred'][n][idx] for n in SPECIES[species]}, both


CURVE, PER_SPEC, BOTH = {}, {}, {}
for s in SPECIES:
    CURVE[s], PER_SPEC[s], BOTH[s] = mean_curves(s, CALLS[s].index)
STD = {s: pz.standardize_rows(CURVE[s]) for s in SPECIES}


def specimen_peaks(s, rows):
    out = {}
    for n in SPECIES[s]:
        c = PER_SPEC[s][n][rows]
        finite = np.isfinite(c).all(0)
        out[n] = GRID_UNITS[s][finite][np.argmax(c[:, finite], axis=1)]
    return out


def pair_separated(s, a, b):
    """H4b's rule for one gene pair: per-specimen peaks >= the species' threshold apart, a before b in both."""
    C = CALLS[s]
    ia, ib = C.index.get_loc(a), C.index.get_loc(b)
    sp = specimen_peaks(s, [ia, ib])
    d = [sp[n][1] - sp[n][0] for n in SPECIES[s]]
    return all(x >= UNIT_SEP[s] for x in d), d


H0_ROWS = []
for a, b in (('Slc5a2', 'Slc5a1'), ('Slc7a9', 'Slc3a1')):
    C = CALLS['mouse']
    if a in C.index and b in C.index:
        ok, d = pair_separated('mouse', a, b)
        H0_ROWS.append({'species': 'mouse', 'pair': f'{a} → {b}', 'peak a': C.loc[a, 'peak'], 'peak b': C.loc[b, 'peak'],
                        'Δ per specimen': ', '.join(f'{x:.2f}' for x in d), 'zonated a/b': f'{C.loc[a, "zonated"]}/{C.loc[b, "zonated"]}',
                        'separated': ok})
    else:
        H0_ROWS.append({'species': 'mouse', 'pair': f'{a} → {b}', 'separated': False, 'note': 'not PT-expressed'})
for g in ('SLC5A2', 'SLC5A1'):
    C = CALLS['human']
    H0_ROWS.append({'species': 'human', 'pair': g, 'label peak': C.loc[g, 'label_peak'] if g in C.index else 'not PT-expressed',
                    'zonated a/b': str(C.loc[g, 'zonated']) if g in C.index else ''})
H0 = pd.DataFrame(H0_ROWS)
H0_PASS = bool(H0[H0.species.eq('mouse')].separated.all())
display(H0)
print('H0 passes (mouse):', H0_PASS)
H0.to_csv(tables / 'h0_positive_control.csv', index=False)


def quintile_strata(values, eligible):
    out = np.full(len(values), -1)
    v = pd.Series(np.asarray(values, float)[eligible])
    out[eligible] = pd.qcut(v.rank(method='first'), 5, labels=False).to_numpy()
    return out


H4_ROWS, H4_PAIRS = [], {}
rng = np.random.default_rng(SEED)
for s in SPECIES:
    C = CALLS[s]
    elig = (C.zonated & C.probe_clean.astype(bool) & C.component.ge(0)).to_numpy()
    fam = C.component.fillna(-1).astype(int).to_numpy()
    members = pd.Series(np.arange(len(C)))[elig].groupby(fam[elig]).apply(list)
    members = members[members.map(len) >= 2]
    pairs = np.array([(i, j) for g in members for k, i in enumerate(g) for j in g[k + 1:]], int).reshape(-1, 2)
    row = {'species': s, 'families with ≥ 2 zonated paralogs': len(members), 'pairs': len(pairs)}
    if len(members) < MIN_FAMILIES:
        row['reading'] = 'not run (stop rule)'
        H4_ROWS.append(row)
        continue
    strata = quintile_strata(C.log2_cp10k, elig) * 5 + quintile_strata(C.Z, elig)
    a, b = pz.matched_pair_draws(pairs, strata, fam, elig, N_DRAWS, rng)
    peaks = C.peak.to_numpy()
    sep, dis = pz.pair_statistics(peaks, STD[s], pairs[:, 0], pairs[:, 1])
    nsep, ndis = pz.pair_statistics(peaks, STD[s], a, b)
    p_sep = pz.global_two_sided(sep.mean(), nsep.mean(axis=0))
    p_dis = pz.global_two_sided(dis.mean(), ndis.mean(axis=0))
    pct_sep = (nsep <= sep[:, None]).mean(axis=1)
    pct_dis = (ndis <= dis[:, None]).mean(axis=1)
    family_of = fam[pairs[:, 0]]
    fam_pct = pd.DataFrame({'family': family_of, 'sep': pct_sep, 'dis': pct_dis}).groupby('family').mean()
    w_sep, w_dis = wilcoxon(fam_pct.sep - .5), wilcoxon(fam_pct.dis - .5)
    below = sep.mean() < nsep.mean() and dis.mean() < ndis.mean()
    above = sep.mean() > nsep.mean() and dis.mean() > ndis.mean()
    reading = ('co-position' if below and max(p_sep, p_dis) < ALPHA else
               'partition' if above and max(p_sep, p_dis) < ALPHA else 'neither')
    row.update({'mean separation': sep.mean(), 'null separation': nsep.mean(), 'p separation': p_sep,
                'mean 1 − r': dis.mean(), 'null 1 − r': ndis.mean(), 'p 1 − r': p_dis,
                'family percentile separation (median)': fam_pct.sep.median(), 'Wilcoxon p (separation)': w_sep.pvalue,
                'family percentile 1 − r (median)': fam_pct.dis.median(), 'Wilcoxon p (1 − r)': w_dis.pvalue,
                'reading': reading})
    H4_ROWS.append(row)
    primary(f'H4 {s}: paralog peak separation against matched pairs', p_sep, f'{sep.mean():.3f} vs {nsep.mean():.3f}')
    primary(f'H4 {s}: paralog shape dissimilarity against matched pairs', p_dis, f'{dis.mean():.3f} vs {ndis.mean():.3f}')
    genes = C.index.to_numpy()
    sp = specimen_peaks(s, pairs.ravel())
    pa = {n: v[0::2] for n, v in sp.items()}
    pb = {n: v[1::2] for n, v in sp.items()}
    d1, d2 = (pa[n] - pb[n] for n in SPECIES[s])
    separated = (np.abs(d1) >= UNIT_SEP[s]) & (np.abs(d2) >= UNIT_SEP[s]) & (np.sign(d1) == np.sign(d2))
    lab = {n: C[f'label_peak_{n}'].to_numpy() for n in SPECIES[s]}
    label_sep = np.logical_and.reduce([lab[n][pairs[:, 0]] != lab[n][pairs[:, 1]] for n in SPECIES[s]])
    H4_PAIRS[s] = pd.DataFrame({'family': family_of, 'a': genes[pairs[:, 0]], 'b': genes[pairs[:, 1]],
                                'peak a': peaks[pairs[:, 0]], 'peak b': peaks[pairs[:, 1]], 'separation': sep,
                                '1 − r': dis, 'percentile separation': pct_sep, 'percentile 1 − r': pct_dis,
                                'separated (both specimens)': separated, 'label peaks differ (both specimens)': label_sep})
H4 = pd.DataFrame(H4_ROWS)
display(H4.round(4))
H4.to_csv(tables / 'h4_copositioning.csv', index=False)
for s, t in H4_PAIRS.items():
    t.to_csv(tables / f'h4_pairs_{s}.csv', index=False)
    sepd = t[t['separated (both specimens)']]
    print(f"H4b {s}: {len(sepd)} of {len(t)} pairs separate in both specimens; label peaks also differ for "
          f"{int(sepd['label peaks differ (both specimens)'].sum())}")
    display(sepd.sort_values('separation', ascending=False).head(25).round(2))
    curve_only = int((t['separated (both specimens)'] & ~t['label peaks differ (both specimens)']).sum())
    label_only = int((~t['separated (both specimens)'] & t['label peaks differ (both specimens)']).sum())
    print(f"H4b' {s}: separated on the curve {int(t['separated (both specimens)'].sum())}, label peaks differ "
          f"{int(t['label peaks differ (both specimens)'].sum())}; curve only {curve_only}, labels only {label_only}")
    if s == 'mouse':
        C = CALLS[s]
        seg = C.peak_segment
        zon = C[C.zonated & C.probe_clean.astype(bool)]
        marg = zon.peak_segment.value_counts(normalize=True)
        same = (seg.reindex(t.a).to_numpy() == seg.reindex(t.b).to_numpy()).mean()
        print(f"D10: zonated paralog pairs in the same peak segment {same:.2f}; chance {float((marg ** 2).sum()):.2f}")

### H4c · How separated mouse paralogs hand over (85's binned edges, U_PT)

In [ ]:
DM = W['mouse']['data']
BINS, XB = {}, {}
for n in MOUSE:
    rows = np.flatnonzero(DM['specimen'] == n)
    b = ts.equal_count_bins(DM['position'][rows], N_BINS)
    BINS[n] = (rows, b)
    XB[n] = to_mouse(np.array([np.median(DM['position'][rows][b == k]) for k in range(N_BINS)]))

Q3 = {}
for n in MOUSE:
    rows, b = BINS[n]
    seg = DM['segment'][rows]
    for boundary, later in ((1, ('PT-S2', 'PT-S3')), (2, ('PT-S3',))):
        frac = np.bincount(b, weights=np.isin(seg, later).astype(float), minlength=N_BINS) / np.bincount(b, minlength=N_BINS)
        _, e = ts.fit_edges(XB[n], frac[None], flat=.1)
        e = e[e.direction > 0]
        Q3[(n, boundary)] = float(e.loc[(e.position - boundary).abs().idxmin(), 'width']) if len(e) else np.nan
print('Q3 label-step widths on U_PT bins:', {k: round(v, 3) for k, v in Q3.items()})


def ratio_profile(n, a, b):
    rows, bins = BINS[n]
    cols = PANEL['mouse'].get_indexer([a, b])
    r = ts.binned_log_rates(DM['counts'][rows][:, cols], DM['library'][rows], bins, N_BINS)
    y = r[0] - r[1]
    return y - y.mean()


H4C_ROWS = []
if 'mouse' in H4_PAIRS:
    for _, p in H4_PAIRS['mouse'][H4_PAIRS['mouse']['separated (both specimens)']].iterrows():
        fits = {n: ts.fit_edges(XB[n], ratio_profile(n, p.a, p.b)[None]) for n in MOUSE}
        ea, eb = (fits[n][1].reset_index(drop=True) for n in MOUSE)
        ag = ts.agreed_edges(ea, eb) if len(ea) and len(eb) else pd.DataFrame()
        row = {'a': p.a, 'b': p.b, 'family': p.family, 'peak a': p['peak a'], 'peak b': p['peak b']}
        if len(ag):
            e = ag.loc[ag.amplitude.abs().idxmax()]
            row.update({'edges agreed': len(ag), 'position': e.position, 'width': e.width, 'width A': e['width A'],
                        'width B': e['width B'], 'amplitude': e.amplitude, 'class': e['class'], 'location': e.location})
        else:
            row.update({'edges agreed': 0, 'class': 'no agreed edge'})
        H4C_ROWS.append(row)
H4C = pd.DataFrame(H4C_ROWS)
if len(H4C):
    display(H4C.round(2))
    display(pd.crosstab(H4C['class'], H4C.get('location', pd.Series('', index=H4C.index)).fillna('')))
H4C.to_csv(tables / 'h4c_handovers.csv', index=False)

## 8 · H5–H6 · Sequence divergence and genomic distance against positional similarity

In [ ]:
GENE_POS = {}
for s in SPECIES:
    iv = pz.gene_intervals(pz.transcript_table(CDNA[s]))
    iv['mid'] = (iv.start + iv.end) / 2
    GENE_POS[s] = iv


def paralog_pairs(s):
    C = CALLS[s]
    ok = C.probe_clean.astype(bool)
    ids = INV[s].gene_id.astype(str).str.split('.').str[0]
    by_id = pd.Series(ids.index, index=ids.to_numpy())
    by_id = by_id[~by_id.index.duplicated()]
    t = pd.read_csv(PARALOGS[s], sep='\t', usecols=['gene_stable_id', 'homology_gene_stable_id', 'homology_type',
                                                     'identity', 'homology_identity', 'ds'])
    t['a'] = t.gene_stable_id.map(by_id)
    t['b'] = t.homology_gene_stable_id.map(by_id)
    t = t.dropna(subset=['a', 'b'])
    keep = t.a.isin(C.index[ok]) & t.b.isin(C.index[ok]) & t.a.ne(t.b)
    t = t[keep].copy()
    t['identity_mean'] = (t.identity.astype(float) + t.homology_identity.astype(float)) / 2
    t['key'] = [tuple(sorted(x)) for x in zip(t.a, t.b)]
    t = t.sort_values('identity_mean', ascending=False).drop_duplicates('key')
    t['a'], t['b'] = zip(*t.key)
    zon = C.zonated
    t = t[zon.reindex(t.a).to_numpy() | zon.reindex(t.b).to_numpy()].copy()
    idx = C.index.get_indexer
    t['r'] = np.einsum('ij,ij->i', STD[s][idx(t.a)], STD[s][idx(t.b)])
    t['ds'] = pd.to_numeric(t.ds, errors='coerce')
    comp = C.component.fillna(-1).astype(int)
    t['same_orthogroup'] = (comp.reindex(t.a).to_numpy() == comp.reindex(t.b).to_numpy()) & (comp.reindex(t.a).to_numpy() >= 0)
    gid = INV[s].gene_id.astype(str).str.split('.').str[0]
    pos = GENE_POS[s].reindex(gid.reindex(t.a).to_numpy()), GENE_POS[s].reindex(gid.reindex(t.b).to_numpy())
    same_chrom = pos[0].chrom.to_numpy() == pos[1].chrom.to_numpy()
    t['distance'] = np.where(same_chrom, np.abs(pos[0].mid.to_numpy() - pos[1].mid.to_numpy()), np.inf)
    t['tandem'] = t.distance < TANDEM
    t['position known'] = pos[0].chrom.notna().to_numpy() & pos[1].chrom.notna().to_numpy()
    # Compara components of the retained pairs, for the cluster bootstrap
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for x, y in zip(t.a, t.b):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry
    t['cluster'] = [find(x) for x in t.a]
    return t.drop(columns='key').reset_index(drop=True)


def cluster_bootstrap(t, x, y, n=2000, seed=SEED):
    rng_b = np.random.default_rng(seed)
    groups = [g.index.to_numpy() for _, g in t.groupby('cluster')]
    out = []
    for _ in range(n):
        pick = rng_b.integers(0, len(groups), len(groups))
        rows = np.concatenate([groups[k] for k in pick])
        out.append(spearmanr(t.loc[rows, x], t.loc[rows, y]).statistic)
    return np.nanpercentile(out, [2.5, 97.5])


H5_ROWS, H5_PAIRS = [], {}
for s in SPECIES:
    t = paralog_pairs(s)
    H5_PAIRS[s] = t
    t.to_csv(tables / f'h5_paralog_pairs_{s}.csv', index=False)
    for label, sub, x in (('primary: all Ensembl paralog pairs', t, 'identity_mean'),
                          ('within_species_paralog only', t[t.homology_type.eq('within_species_paralog')], 'identity_mean'),
                          ('same HCOP orthogroup only', t[t.same_orthogroup], 'identity_mean'),
                          ('dS < 2 (−dS)', t[t.ds < 2].assign(neg_ds=lambda u: -u.ds), 'neg_ds')):
        if len(sub) < 10:
            H5_ROWS.append({'species': s, 'setting': label, 'pairs': len(sub)})
            continue
        rho = spearmanr(sub[x], sub.r)
        one = rho.pvalue / 2 if rho.statistic > 0 else 1 - rho.pvalue / 2
        lo, hi = cluster_bootstrap(sub.reset_index(drop=True), x, 'r')
        H5_ROWS.append({'species': s, 'setting': label, 'pairs': len(sub), 'clusters': sub.cluster.nunique(),
                        'Spearman rho': rho.statistic, 'one-sided p': one, 'cluster 95% low': lo, 'cluster 95% high': hi,
                        'median r': sub.r.median()})
    # floor: matched random pairs among PT-expressed probe-clean genes
    C = CALLS[s]
    elig = C.probe_clean.astype(bool).to_numpy()
    strata = quintile_strata(C.log2_cp10k, elig) * 5 + quintile_strata(C.Z, elig)
    idx = C.index.get_indexer
    obs_pairs = np.column_stack([idx(t.a), idx(t.b)])
    ra, rb = pz.matched_pair_draws(obs_pairs, strata, C.component.fillna(-1).astype(int).to_numpy(), elig, 20, rng)
    H5_ROWS.append({'species': s, 'setting': 'floor: matched random pairs', 'pairs': ra.size,
                    'median r': float(np.median(np.einsum('ijk,ijk->ij', STD[s][ra], STD[s][rb])))})
H5 = pd.DataFrame(H5_ROWS)
display(H5.round(4))
H5.to_csv(tables / 'h5_identity.csv', index=False)
h5 = H5[(H5.species == 'mouse') & H5.setting.str.startswith('primary')].iloc[0]
primary('H5 mouse: identity vs positional similarity (Spearman, one-sided)', h5['one-sided p'], f"rho {h5['Spearman rho']:.3f}")
H5_HOLDS = bool(h5['Spearman rho'] > 0 and h5['one-sided p'] < ALPHA and h5['cluster 95% low'] > 0)

H6_ROWS = []
for s in SPECIES:
    t = H5_PAIRS[s][H5_PAIRS[s]['position known']]
    tan, oth = t[t.tandem], t[~t.tandem]
    mw6 = mannwhitneyu(tan.r, oth.r, alternative='greater') if len(tan) and len(oth) else None
    rho6 = spearmanr(tan.distance, tan.r) if len(tan) >= 10 else None
    one6 = (rho6.pvalue / 2 if rho6.statistic < 0 else 1 - rho6.pvalue / 2) if rho6 is not None else np.nan
    rr = pd.DataFrame({'r': t.r.rank(), 'identity': t.identity_mean.rank(), 'tandem': t.tandem.astype(float)})
    X = np.column_stack([np.ones(len(rr)), rr.identity, rr.tandem])
    beta, *_ = np.linalg.lstsq(X, rr.r, rcond=None)
    resid = rr.r - X @ beta
    se = np.sqrt(np.diag(np.linalg.inv(X.T @ X)) * (resid @ resid) / (len(rr) - 3))
    H6_ROWS.append({'species': s, 'pairs with positions': len(t), 'tandem': len(tan), 'median r tandem': tan.r.median(),
                    'median r other': oth.r.median(), 'MW p (tandem > other)': mw6.pvalue if mw6 else np.nan,
                    'Spearman distance vs r (tandem)': rho6.statistic if rho6 is not None else np.nan,
                    'one-sided p (falls with distance)': one6,
                    'rank regression t(identity)': beta[1] / se[1], 'rank regression t(tandem)': beta[2] / se[2]})
    if s == 'mouse':
        primary('H6 mouse: tandem paralogs more similar (MW, one-sided)', H6_ROWS[-1]['MW p (tandem > other)'],
                f"{tan.r.median():.2f} vs {oth.r.median():.2f}")
        primary('H6 mouse: similarity falls with distance among tandem pairs (Spearman, one-sided)', one6,
                f"rho {H6_ROWS[-1]['Spearman distance vs r (tandem)']:.3f}")
H6 = pd.DataFrame(H6_ROWS)
display(H6.round(4))

H6B_ROWS = []
for s in SPECIES:
    C = CALLS[s]
    ok = C[C.probe_clean.astype(bool)]
    gid_s = INV[s].gene_id.astype(str).str.split('.').str[0].reindex(ok.index)
    pos = GENE_POS[s].reindex(gid_s.to_numpy())
    frame = pd.DataFrame({'gene': ok.index, 'chrom': pos.chrom.to_numpy(), 'mid': pos.mid.to_numpy(),
                          'comp': ok.component.fillna(-1).astype(int).to_numpy(), 'zon': ok.zonated.to_numpy()}).dropna(subset=['chrom'])
    para = set(map(tuple, pd.read_csv(PARALOGS[s], sep='\t', usecols=['gene_stable_id', 'homology_gene_stable_id']).to_numpy()))
    idmap = dict(zip(ok.index, gid_s))
    nb = []
    for chrom, g in frame.sort_values('mid').groupby('chrom'):
        mids, genes, comps, zons = g.mid.to_numpy(), g.gene.to_numpy(), g.comp.to_numpy(), g.zon.to_numpy()
        for i in range(len(g)):
            j = i + 1
            while j < len(g) and mids[j] - mids[i] < TANDEM:
                if (zons[i] or zons[j]) and not (comps[i] >= 0 and comps[i] == comps[j]) \
                        and (idmap[genes[i]], idmap[genes[j]]) not in para and (idmap[genes[j]], idmap[genes[i]]) not in para:
                    nb.append((genes[i], genes[j], mids[j] - mids[i]))
                j += 1
    nb = pd.DataFrame(nb, columns=['a', 'b', 'distance'])
    idx = C.index.get_indexer
    nb['r'] = np.einsum('ij,ij->i', STD[s][idx(nb.a)], STD[s][idx(nb.b)]) if len(nb) else []
    tan = H5_PAIRS[s][H5_PAIRS[s].tandem]
    mw6b = mannwhitneyu(tan.r, nb.r) if len(tan) and len(nb) else None
    H6B_ROWS.append({'species': s, 'tandem paralog pairs': len(tan), 'neighbour pairs': len(nb),
                     'median r paralogs': tan.r.median(), 'median r neighbours': nb.r.median(),
                     'difference': tan.r.median() - nb.r.median(), 'MW p (two-sided)': mw6b.pvalue if mw6b else np.nan})
    nb.to_csv(tables / f'h6b_neighbour_pairs_{s}.csv', index=False)
H6B = pd.DataFrame(H6B_ROWS)
display(H6B.round(4))
H6B.to_csv(tables / 'h6b_neighbours.csv', index=False)
primary('H6b mouse: tandem paralogs vs non-paralog neighbours (MW, two-sided)', H6B.iloc[0]['MW p (two-sided)'],
        f"{H6B.iloc[0]['median r paralogs']:.2f} vs {H6B.iloc[0]['median r neighbours']:.2f}")
H6.to_csv(tables / 'h6_genomic_distance.csv', index=False)

## 9 · H7 · Family units across species, with the 1:1 analysis's own test (U_all)

Each non-1:1 orthogroup's family-clean members are summed in each species. 71's model, denominator and moderation
run unchanged; the variance prior is estimated over 71's genes and the family units together.

In [ ]:
from scipy.stats import false_discovery_control

from pseudospace.aki_along_pt import centre_rows
from pseudospace.pathway_pipelines import (GeneModel, moderated_f, partition_groups, specimen_shape_lr,
                                           standard_gene_model)


def family_members(component, species, *, clean='family_clean'):
    inv = INV[species]
    m = inv[(inv.component == component) & inv[clean].astype(bool)]
    return [g for g in m.index if not g.upper().startswith('MT-')]


comps = sorted(set(INV['mouse'].component[INV['mouse'].non]) | set(INV['human'].component[INV['human'].non]))
FAMILIES = {}
for comp in comps:
    mm, hh = family_members(comp, 'mouse'), family_members(comp, 'human')
    if mm and hh:
        FAMILIES[int(comp)] = {'mouse': mm, 'human': hh}
print(f'{len(comps):,} non-1:1 orthogroups; {len(FAMILIES):,} with a family-clean member in each species')


def family_matrix(units_order='inp'):
    """Units (INP order) x families: each species' family sums from its native counts."""
    names = list(FAMILIES)
    out = sparse.lil_matrix((len(POSITION), len(names)))
    blocks = []
    for s in SPECIES:
        x = no.family_sums(NATIVE[s]['counts'], PANEL[s], {k: FAMILIES[k][s] for k in names})
        blocks.append((NATIVE[s]['rows'], x))
    rows = np.concatenate([r for r, _ in blocks])
    stacked = sparse.vstack([x for _, x in blocks]).tocsr()
    order = np.argsort(rows)
    assert (rows[order] == np.arange(len(POSITION))).all()
    return stacked[order], names


XF, FNAMES = family_matrix()
det_f = np.mean([np.asarray((XF[SPECIMEN == n] > 0).mean(axis=0)).ravel() for n in MOUSE + HUMAN], axis=0)
F_ELIG = det_f >= DETECTION
XF, FNAMES = XF[:, np.flatnonzero(F_ELIG)], [f for f, e in zip(FNAMES, F_ELIG) if e]
FAM_LABEL = {k: f"{'/'.join(FAMILIES[k]['mouse'][:3])}{'…' if len(FAMILIES[k]['mouse']) > 3 else ''} | "
                f"{'/'.join(FAMILIES[k]['human'][:3])}{'…' if len(FAMILIES[k]['human']) > 3 else ''}" for k in FAMILIES}
print(f'{len(FNAMES):,} family units pass the 2% equal-specimen detection on U_all')

PARTS = partition_groups(SPECIMEN, INP['human'])
LABELS = list(PARTS)
SHORT = dict(zip(LABELS, ['condition', 'relabeling 1', 'relabeling 2']))


def slug(label):
    return label.replace(':', '_').replace('+', '_')


def model_payload(model):
    out = {f'stat|{c}': model.statistics[c].to_numpy() for c in model.statistics}
    out.update(grid=model.grid, knots=model.knots, reference=model.reference, case=model.case, delta=model.delta,
               se=model.se, alpha=np.asarray(model.dispersion['alpha'], float),
               alpha_converged=np.asarray(model.dispersion['converged'], bool))
    for name, curve in model.specimen_curves.items():
        out[f'curve|{name}'] = curve
        out[f'converged|{name}'] = model.specimen_converged[name].to_numpy()
        out[f'separated|{name}'] = model.specimen_separated[name].to_numpy()
    return out


def fit_partitions(tag, counts, library, position, specimen, parts, genes):
    key = {'counts': digest(counts), 'library': library, 'position': position, 'specimen': specimen, 'genes': genes}
    models, denom = {}, {}
    for label, (group, nuisance) in parts.items():
        alpha = None if label == LABELS_OF[tag][0] else {'alpha': models[LABELS_OF[tag][0]]['alpha'],
                                                          'converged': models[LABELS_OF[tag][0]]['alpha_converged'].astype(bool)}
        models[label] = cached(f'gene_model_{tag}_{slug(label)}', lambda group=group, nuisance=nuisance, alpha=alpha:
                               model_payload(standard_gene_model(counts, library, position, group, specimen, genes=genes,
                                                                 nuisance_shape=nuisance, dispersion=alpha,
                                                                 basis_df=BASIS_DF, n_jobs=N_JOBS)),
                               params={'basis_df': BASIS_DF, 'label': label},
                               inputs={**key, 'group': group, 'nuisance': nuisance,
                                       'alpha': None if alpha is None else alpha['alpha']})
        sp = models[LABELS_OF[tag][0]]
        sp_alpha = {'alpha': sp['alpha'], 'converged': sp['alpha_converged'].astype(bool)}
        denom[label] = cached(f'specimen_shape_{tag}_{slug(label)}', lambda group=group, nuisance=nuisance:
                              specimen_shape_lr(counts, library, position, group, specimen, sp_alpha, nuisance_shape=nuisance,
                                                basis_df=BASIS_DF, n_jobs=N_JOBS),
                              params={'basis_df': BASIS_DF, 'label': label},
                              inputs={**key, 'group': group, 'nuisance': nuisance, 'alpha': sp_alpha['alpha']})
    return models, denom


LABELS_OF = {'families': LABELS}
FM, FD = fit_partitions('families', XF, LIBRARY, POSITION, SPECIMEN, PARTS, FNAMES)
LMC71 = np.log(np.asarray(INP['counts'].mean(axis=0)).ravel())
LMCF = np.log(np.asarray(XF.mean(axis=0)).ravel())
den71 = {label: stored(T71, f'specimen_shape_{DS}_{slug(label)}') for label in LABELS}
mod71 = {label: stored(T71, f'gene_model_{DS}_{slug(label)}') for label in LABELS}
FSTAT, JOINT = {}, {}
for label in LABELS:
    df2 = int(den71[label]['df2'])
    assert int(FD[label]['df2']) == df2
    alone = moderated_f(pd.Series(mod71[label]['stat|T_spatial'], index=GENES71), DF1, den71[label]['LR_spec'], df2,
                        covariate=LMC71)
    np.testing.assert_allclose(alone.F.to_numpy(), STAT71[SHORT[label]].F.to_numpy(), rtol=1e-6)
    num = pd.Series(np.concatenate([mod71[label]['stat|T_spatial'], FM[label]['stat|T_spatial']]),
                    index=[f'g|{g}' for g in GENES71] + [f'f|{k}' for k in FNAMES])
    joint = moderated_f(num, DF1, np.concatenate([den71[label]['LR_spec'], FD[label]['LR_spec']]), df2,
                        covariate=np.concatenate([LMC71, LMCF]))
    joint['q_joint'] = false_discovery_control(joint.p.to_numpy())
    JOINT[label] = joint
    f = joint.loc[[f'f|{k}' for k in FNAMES]].copy()
    f.index = FNAMES
    f['q'] = false_discovery_control(f.p.to_numpy())
    f['converged'] = FM[label]['stat|converged'].astype(bool) & FD[label]['converged'].astype(bool)
    f['separated'] = FM[label]['stat|separated'].astype(bool) | FD[label]['separated'].astype(bool)
    FSTAT[label] = f
GUARDS.append("H7: 71's moderated F reproduced from its cached T_spatial and LR_spec (rtol 1e-6) for all three partitions")
rel_p = np.concatenate([FSTAT[l].p.to_numpy() for l in LABELS[1:]])
F_CAL = float(np.mean(rel_p <= ALPHA))
cond = FSTAT['species']
fam_table = pd.DataFrame({'family': FNAMES, 'label': [FAM_LABEL[k] for k in FNAMES],
                          'mouse genes': [', '.join(FAMILIES[k]['mouse']) for k in FNAMES],
                          'human genes': [', '.join(FAMILIES[k]['human']) for k in FNAMES],
                          'q (families)': cond.q.to_numpy(), 'q (joint)': JOINT['species'].loc[[f'f|{k}' for k in FNAMES], 'q_joint'].to_numpy(),
                          'p': cond.p.to_numpy(), 'z': cond.z.to_numpy(),
                          'p relabeling 1': FSTAT[LABELS[1]].p.to_numpy(), 'p relabeling 2': FSTAT[LABELS[2]].p.to_numpy(),
                          'converged': cond.converged.to_numpy(), 'log mean count': LMCF})
fam_table['called'] = fam_table['q (joint)'] <= ALPHA

hs = UNITS.var.human_symbol.astype(str)
one_to_one = (INV['mouse'].class_support3.reindex(GENES71).eq('1:1').to_numpy()
              & INV['human'].class_support3.reindex(hs.reindex(GENES71).to_numpy()).eq('1:1').to_numpy())
comp_table = pd.DataFrame({'called': JOINT['species'].loc[[f'g|{g}' for g in GENES71], 'q_joint'].to_numpy()[one_to_one] <= ALPHA,
                           'lmc': LMC71[one_to_one], 'family': False})
both = pd.concat([comp_table, pd.DataFrame({'called': fam_table.called, 'lmc': LMCF, 'family': True})], ignore_index=True)
strata7 = pd.qcut(both.lmc.rank(method='first'), 10, labels=False).to_numpy()
mh7 = pf.mantel_haenszel(both.family.to_numpy(), both.called.to_numpy(), strata7)
p7 = 2 * norm.sf(abs(mh7['z']))
H7 = pd.Series({'family units': len(FNAMES), 'called (q joint ≤ 0.05)': int(fam_table.called.sum()),
                'share called': fam_table.called.mean(), '1:1 comparator genes': int(one_to_one.sum()),
                '1:1 share called': comp_table.called.mean(), 'OR_MH': mh7['or_mh'], 'two-sided p': p7,
                'relabeled family p ≤ 0.05 (pooled share)': F_CAL,
                'calibration': 'pass' if F_CAL <= CALIBRATION_MAX else 'fail',
                'relabeled calls (q joint ≤ 0.05)': ', '.join(str(int((JOINT[l].loc[[f'f|{k}' for k in FNAMES], 'q_joint'] <= ALPHA).sum())) for l in LABELS[1:])})
display(H7.to_frame('value'))
primary('H7: family units called vs matched 1:1 genes (MH OR, two-sided)', p7, f"OR {mh7['or_mh']:.2f}")

# H7c: divergence against the expansion
fam_table['mouse members'] = [len(FAMILIES[k]['mouse']) for k in FNAMES]
fam_table['human members'] = [len(FAMILIES[k]['human']) for k in FNAMES]
ratio = np.log2(fam_table['mouse members'] / fam_table['human members'])
H7C = pd.Series({'Spearman z vs log2 mouse/human members': spearmanr(fam_table.z, ratio).statistic,
                 'p': spearmanr(fam_table.z, ratio).pvalue,
                 'Spearman z vs total members': spearmanr(fam_table.z, fam_table['mouse members'] + fam_table['human members']).statistic,
                 'p (total)': spearmanr(fam_table.z, fam_table['mouse members'] + fam_table['human members']).pvalue})
display(H7C.to_frame('H7c').round(4))

# H7b: 71 genes inside multi-member orthogroups against their family unit
ok71 = (mod71['species']['stat|converged'].astype(bool) & ~mod71['species']['stat|separated'].astype(bool))
c71, m71 = centre_rows(mod71['species']['delta'], ok71)
cf = FM['species']['delta'] - m71
rows7b = []
fidx = {k: i for i, k in enumerate(FNAMES)}
for i, g in enumerate(GENES71):
    comp_m = INV['mouse'].component.get(g, -1)
    multi = INV['mouse'].class_support3.get(g) in NON or INV['human'].class_support3.get(hs.get(g)) in NON
    if not multi or comp_m not in fidx:
        continue
    j = fidx[comp_m]
    a, b = c71[i], cf[j]
    rows7b.append({'gene': g, 'human': hs.get(g), 'family': comp_m, 'family label': FAM_LABEL[comp_m],
                   '71 q (joint)': JOINT['species'].loc[f'g|{g}', 'q_joint'], 'family q (joint)': fam_table.loc[j, 'q (joint)'],
                   'r(Δ gene, Δ family)': float(np.corrcoef(a, b)[0, 1]) if np.isfinite(a).all() and np.isfinite(b).all() else np.nan})
H7B = pd.DataFrame(rows7b)
if len(H7B):
    H7B['gene called'] = H7B['71 q (joint)'] <= ALPHA
    H7B['family called'] = H7B['family q (joint)'] <= ALPHA
    display(pd.crosstab(H7B['gene called'], H7B['family called']))
    display(H7B[H7B['gene called'] != H7B['family called']].sort_values('71 q (joint)').head(30).round(3))
fam_table.to_csv(tables / 'h7_family_units.csv', index=False)
H7.to_frame('value').to_csv(tables / 'h7_summary.csv')
H7B.to_csv(tables / 'h7b_one_to_one_calls_in_families.csv', index=False)
display(fam_table[fam_table.called].sort_values('p').head(30)[['label', 'q (joint)', 'p relabeling 1', 'p relabeling 2']].round(4))

## 10 · H8 · The single-copy gene against its co-orthologs (U_PT, both species' own fits)

In [ ]:
CROSS = BOTH['mouse'] & BOTH['human']


def cross_curve(s, genes):
    rows = CALLS[s].index.get_indexer(genes)
    return np.mean([PER_SPEC[s][n][rows][:, CROSS] for n in SPECIES[s]], axis=0)


def corr(a, b):
    a, b = np.atleast_2d(a), np.atleast_2d(b)
    return pz.rowwise_pearson(a, b)


H8_FAM = []
for comp in comps:
    sides = {}
    for s in SPECIES:
        inv = INV[s]
        members = [g for g in inv.index[inv.component == comp] if g in CALLS[s].index]
        sides[s] = members
    for S, E in (('mouse', 'human'), ('human', 'mouse')):
        s_clean = [g for g in sides[S] if CALLS[S].loc[g, 'probe_clean']]
        e_clean = [g for g in sides[E] if INV[E].loc[g, 'family_clean']]
        if len(sides[S]) == 1 and len(s_clean) == 1 and len(e_clean) >= 2 and CALLS[S].loc[s_clean[0], 'zonated']:
            H8_FAM.append({'family': int(comp), 'S species': S, 'S gene': s_clean[0], 'E members': e_clean})
print(f'H8: {len(H8_FAM)} families with a zonated single-copy gene and ≥ 2 PT-expressed co-orthologs')

ESUM = {}
for E in SPECIES:
    fams = {f['family']: f['E members'] for f in H8_FAM if f['S species'] != E}
    if not fams:
        continue
    X = no.family_sums(NATIVE[E]['counts'], PANEL[E], fams)
    w = w_run(E, columns=(X, [f'sum|{k}' for k in fams]), shapes=W[E]['z']['shape'], tag=f'{E}_family_sums')
    rows = W[E]['genes']
    zc = w['z']['centred']
    ESUM[E] = {k: np.mean([zc[n][i][CROSS] for n in SPECIES[E]], axis=0) for i, k in enumerate(fams)}
    ESUM[E + '_calls'] = w['calls']

hm = pd.read_csv(COMPARA_HM, sep='\t', usecols=['gene_stable_id', 'homology_gene_stable_id', 'identity', 'homology_identity'])
hm['identity_mean'] = (hm.identity + hm.homology_identity) / 2
hm_id = hm.set_index(['gene_stable_id', 'homology_gene_stable_id']).identity_mean
gid = {s: INV[s].gene_id.astype(str).str.split('.').str[0] for s in SPECIES}
kept_partner = {'mouse': hs, 'human': pd.Series(hs.index, index=hs.to_numpy())}
rows8 = []
for f in H8_FAM:
    S, E = f['S species'], ('human' if f['S species'] == 'mouse' else 'mouse')
    s_curve = cross_curve(S, [f['S gene']])[0]
    e_curves = cross_curve(E, f['E members'])
    r_each = corr(np.repeat(s_curve[None], len(e_curves), 0), e_curves)
    r_sum = float(corr(s_curve, ESUM[E][f['family']])[0])
    best = int(np.nanargmax(r_each))
    zon = CALLS[E].reindex(f['E members']).zonated.to_numpy()
    h, mse = (f['S gene'], f['E members']) if S == 'human' else (None, None)
    ident = []
    for g in f['E members']:
        key = (gid['human'][f['S gene']], gid['mouse'][g]) if S == 'human' else (gid['human'][g], gid['mouse'][f['S gene']])
        ident.append(float(hm_id.get(key, np.nan)) if not isinstance(hm_id.get(key, np.nan), pd.Series) else float(hm_id.get(key).max()))
    ident = np.array(ident)
    partner = kept_partner[S].get(f['S gene'])
    rows8.append({'family': f['family'], 'S species': S, 'S gene': f['S gene'], 'S Z': CALLS[S].loc[f['S gene'], 'Z'],
                  'E members': ', '.join(f['E members']), 'k': len(f['E members']), 'E zonated': int(zon.sum()),
                  'best E (position)': f['E members'][best], 'r best E': float(r_each[best]), 'r E sum': r_sum,
                  'sum beats best member': r_sum > np.nanmax(r_each),
                  'sequence-closest E': f['E members'][int(np.nanargmax(ident))] if np.isfinite(ident).any() else '',
                  'kept partner': partner if partner in f['E members'] else '',
                  'r each': ', '.join(f'{g} {v:.2f}' for g, v in zip(f['E members'], r_each))})
H8 = pd.DataFrame(rows8)
if len(H8):
    sub = H8[H8['E zonated'] >= 2]
    h8a = binomtest(int(sub['sum beats best member'].sum()), len(sub), .5) if len(sub) else None
    print(f"H8a: E sum beats the best member in {int(sub['sum beats best member'].sum())} of {len(sub)} families with ≥ 2 "
          f"zonated co-orthologs; sign test p {h8a.pvalue if h8a else np.nan:.3g}")
    if h8a:
        primary('H8a: family sum matches the single-copy gene better than its best member (sign test)', h8a.pvalue,
                f"{int(sub['sum beats best member'].sum())}/{len(sub)}")
    # H8b: comparator 1:1 pairs (export), same-species gene zonated, matched on its Z quintile
    comp_rows = []
    for g in UNITS.var_names:
        h = hs.get(g)
        if g in CALLS['mouse'].index and h in CALLS['human'].index:
            if INV['mouse'].class_support3.get(g) == '1:1' and INV['human'].class_support3.get(h) == '1:1' \
                    and CALLS['mouse'].loc[g, 'probe_clean'] and CALLS['human'].loc[h, 'probe_clean']:
                comp_rows.append((g, h))
    cm = cross_curve('mouse', [g for g, _ in comp_rows])
    ch = cross_curve('human', [h for _, h in comp_rows])
    r11 = corr(cm, ch)
    C11 = pd.DataFrame({'mouse': [g for g, _ in comp_rows], 'human': [h for _, h in comp_rows], 'r': r11})
    pct = []
    for _, f in H8.iterrows():
        S = f['S species']
        zq = pd.qcut(CALLS[S].Z[CALLS[S].zonated].rank(method='first'), 5, labels=False)
        q_f = zq.get(f['S gene'])
        side = C11[S]
        in_q = side.map(zq).eq(q_f)
        ref = C11.r[in_q & side.isin(CALLS[S].index[CALLS[S].zonated])]
        pct.append(float((ref < f['r best E']).mean()) if len(ref) else np.nan)
    H8['percentile among matched 1:1 pairs'] = pct
    ok8 = H8['percentile among matched 1:1 pairs'].dropna()
    h8b = wilcoxon(ok8 - .5, alternative='less') if len(ok8) >= 5 else None
    if h8b:
        primary('H8b: best co-ortholog less similar than matched 1:1 pairs (Wilcoxon, one-sided)', h8b.pvalue,
                f'median percentile {ok8.median():.2f}')
    B = C11.assign(mz=C11.mouse.map(CALLS['mouse'].zonated), hz=C11.human.map(CALLS['human'].zonated))
    B = B[B.mz | B.hz]
    cg = GRID[CROSS]
    half = lambda c: np.where(c[:, cg < .5].mean(axis=1) > c[:, cg >= .5].mean(axis=1), 'early', 'late')   # noqa: E731
    same_half = (half(cross_curve('mouse', B.mouse.tolist())) == half(cross_curve('human', B.human.tolist()))).mean()
    print(f"B: 1:1 pairs with either gene zonated {len(B)}; median cross-species r {B.r.median():.2f}; "
          f"same peak half {same_half:.2f}")
    exp_seq = (1 / H8.k).sum()
    print(f"H8b: median percentile of r(S, best E) among matched 1:1 pairs {ok8.median():.2f} (n {len(ok8)}); "
          f"Wilcoxon one-sided p {h8b.pvalue if h8b else np.nan:.3g}. 1:1 pairs r median {np.nanmedian(r11):.2f}")
    print(f"H8c: best positional member = sequence-closest in {int((H8['best E (position)'] == H8['sequence-closest E']).sum())} "
          f"of {len(H8)} (chance {exp_seq:.1f}); = kept partner in {int((H8['best E (position)'] == H8['kept partner']).sum())} "
          f"of {int(H8['kept partner'].ne('').sum())} with a kept partner among E")
    display(H8.drop(columns=['r each']).round(3))
H8.to_csv(tables / 'h8_single_copy.csv', index=False)

# H8d: dosage sharing in level (within-species percentiles of log2 CP10k among PT-expressed genes)
pctl = {s: CALLS[s].log2_cp10k.rank(pct=True) for s in SPECIES}
rows8d = []
for f in H8_FAM:
    S, E = f['S species'], ('human' if f['S species'] == 'mouse' else 'mouse')
    d = W[E]['data']
    cp = []
    for n in SPECIES[E]:
        mk = d['specimen'] == n
        x_ = no.family_sums(d['counts'][mk], PANEL[E], {0: f['E members']})
        cp.append(float(x_.sum()) / d['library'][mk].sum() * 1e4)
    sum_l2 = np.log2(np.mean(cp))
    sum_pct = float((CALLS[E].log2_cp10k < sum_l2).mean())
    top = max(pctl[E].reindex(f['E members']).dropna())
    rows8d.append({'family': f['family'], 'S species': S, 'S percentile': pctl[S][f['S gene']], 'E top member percentile': top,
                   'E sum percentile': sum_pct})
H8D = pd.DataFrame(rows8d)
if len(H8D):
    a1 = (H8D['S percentile'] > H8D['E top member percentile'])
    a2 = ((H8D['E sum percentile'] - H8D['S percentile']).abs() < (H8D['E top member percentile'] - H8D['S percentile']).abs())
    print(f"H8d: single-copy percentile above the expanded side's top member in {int(a1.sum())}/{len(H8D)} "
          f"(sign test p {binomtest(int(a1.sum()), len(H8D)).pvalue:.3g}); expanded sum closer than its top member in "
          f"{int(a2.sum())}/{len(H8D)} (p {binomtest(int(a2.sum()), len(H8D)).pvalue:.3g})")
H8D.to_csv(tables / 'h8d_dosage_levels.csv', index=False)

## 11 · H9 · Genes without an ortholog (descriptive)

In [ ]:
H9 = pd.concat({s: CALLS[s][CALLS[s].class_support3.isin(['no_ortholog', 'no_consensus']) & CALLS[s].probe_clean.astype(bool)]
                [['class_support3', 'log2_cp10k', 'zonated', 'strong', 'Z', 'peak', 'label_peak']]
                for s in SPECIES}, names=['species', 'gene']).sort_values(['species', 'zonated', 'Z'], ascending=[True, False, False])
display(H9[H9.zonated].round(2))
print(H9.groupby(['species', 'class_support3']).agg(n=('Z', 'size'), zonated=('zonated', 'sum'), strong=('strong', 'sum')))
H9.to_csv(tables / 'h9_no_ortholog.csv')

## 12 · H10 · Agreement with male microdissection (Chen 2021), non-1:1 against 1:1

In [ ]:
raw21 = pd.read_csv(CHEN21, sep='\t')
pat = re.compile(r'^m(\d+)_(PTS[123])_([a-c])$')
cols21 = {c: m_.group(2) for c in raw21.columns if (m_ := pat.match(str(c)))}
tpm21 = raw21.dropna(subset=['Gene_symbol']).set_index('Gene_symbol')[list(cols21)].astype(float)
seg21 = tpm21.T.groupby(pd.Series(cols21)).mean().T
seg21 = seg21.assign(_m=seg21.max(axis=1)).sort_values('_m', ascending=False)
seg21 = seg21[~seg21.index.duplicated()].drop(columns='_m')
chen_peak = seg21.idxmax(axis=1).str.replace('PT', '')
chen_ok = seg21.max(axis=1) >= 1
m = CALLS['mouse']
m['chen21_peak'] = chen_peak.reindex(m.index)
m['chen21_ok'] = chen_ok.reindex(m.index).fillna(False).astype(bool)
H10_ROWS = []
for label, mask in (('strong, probe-clean', m.strong & m.probe_clean.astype(bool)),
                    ('zonated, probe-ambiguous (no claim)', m.zonated & ~m.probe_clean.astype(bool))):
    t = m[mask & m.chen21_ok & m.cls.isin(['1:1', 'non-1:1'])]
    for view in ('peak_segment', 'label_peak'):
        agree = t[view].astype(str).eq(t.chen21_peak)
        tab10 = pd.crosstab(t.cls, agree).reindex(index=['non-1:1', '1:1'], columns=[True, False], fill_value=0)
        fx = fisher_exact(tab10.to_numpy()) if tab10.to_numpy().min(axis=1).sum() >= 0 and len(t) else None
        H10_ROWS.append({'genes': label, 'our view': view, 'non-1:1 agree': f"{tab10.loc['non-1:1', True]}/{tab10.loc['non-1:1'].sum()}",
                         '1:1 agree': f"{tab10.loc['1:1', True]}/{tab10.loc['1:1'].sum()}",
                         'OR': fx.statistic if fx else np.nan, 'two-sided p': fx.pvalue if fx else np.nan})
H10 = pd.DataFrame(H10_ROWS)
display(H10.round(4))
H10.to_csv(tables / 'h10_microdissection.csv', index=False)
primary('H10: curve peak agrees with microdissection, non-1:1 vs 1:1 (Fisher, two-sided)', H10.iloc[0]['two-sided p'],
        f"{H10.iloc[0]['non-1:1 agree']} vs {H10.iloc[0]['1:1 agree']}")

## 13 · H13 · Dominance inside expanded families (descriptive)

In [ ]:
dom = []
for s in SPECIES:
    d = W[s]['data']
    C = CALLS[s]
    for comp, g in C[C.component.ge(0) & C.non].groupby('component'):
        if len(g) < 2:
            continue
        tot = np.asarray(d['counts'][:, PANEL[s].get_indexer(g.index)].sum(axis=0)).ravel()
        dom.append({'species': s, 'family': int(comp), 'members': len(g), 'top member': g.index[int(np.argmax(tot))],
                    'top share': tot.max() / tot.sum()})
H13 = pd.DataFrame(dom)
print(H13.groupby('species')['top share'].describe().round(3))
mw13 = mannwhitneyu(H13['top share'][H13.species.eq('mouse')], H13['top share'][H13.species.eq('human')])
print(f'Mouse vs human top-member share: Mann–Whitney p {mw13.pvalue:.3g} (descriptive)')
H13.to_csv(tables / 'h13_dominance.csv', index=False)

## 14 · H11 · AKI: genes outside the 1:1 pool after injury (76's construction)

G5: 76's inputs are rebuilt by its own construction and its cached model reproduces its table.

In [ ]:
import h5py
from anndata.io import read_elem

UA = read_pt_units(EXPORT['aki'], 'aki')
assert UA.uns['provenance']['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT['aki']
MOUSE_ORTHO = set(orthologs.mouse_symbol.astype(str))
POOL = [g for g in UA.var_names.astype(str) if g in MOUSE_ORTHO]
U_MT = aki_inputs(UA, data_root, orthologs, POOL)['genes']
AK = aki_inputs(UA, data_root, orthologs, U_MT, exclude_genes=MT)
U76 = pd.read_csv(T76 / 'tables' / 'universe.csv').gene.astype(str).tolist()
assert list(AK['genes']) == U76, 'G5: the AKI universe differs from 76'
ST76 = pd.read_csv(T76 / 'tables' / 'gene_statistics.csv', header=[0, 1], index_col=0)
PARTS_A = partition_groups(AK['specimen'], AK['case'])
LABELS_A = list(PARTS_A)
assert LABELS_A == ['species', 'swap:IR2A2+Ctrl1A2', 'swap:IR2A2+Ctrl1A4'], LABELS_A
mod76 = {l: stored(T76, f'gene_model_primary_{slug(l)}') for l in LABELS_A}
den76 = {l: stored(T76, f'specimen_shape_primary_{slug(l)}') for l in LABELS_A}
np.testing.assert_allclose(mod76['species']['stat|T_spatial'], ST76['condition'].T_spatial.to_numpy(), rtol=1e-12)
GUARDS.append(f"G5: 76's universe ({len(U76):,} genes) rebuilt; its cached condition model reproduces its T_spatial")

units_a = AK['obs'].index
XA = sparse.csr_matrix(UA[units_a].X, dtype=float)
cand = [g for g in UA.var_names.astype(str) if g not in MOUSE_ORTHO and not g.startswith('mt-')]
ci = UA.var_names.get_indexer(cand)
det_a = np.mean([np.asarray((XA[AK['specimen'] == n][:, ci] > 0).mean(axis=0)).ravel()
                 for n in CONTROLS + IRS], axis=0)
NEW = [g for g, d_ in zip(cand, det_a) if d_ >= DETECTION]
XN = XA[:, UA.var_names.get_indexer(NEW)]
print(f'H11: {len(NEW)} genes outside the 1:1 pool pass detection (76 reported 468)')
print(INV['mouse'].class_support3.reindex(NEW).value_counts())
LABELS_OF['aki_non'] = LABELS_A
AM, AD = fit_partitions('aki_non', XN, AK['library'], AK['position'], AK['specimen'], PARTS_A, NEW)
LMC76 = np.log(np.asarray(AK['counts'].mean(axis=0)).ravel())
LMCN = np.log(np.asarray(XN.mean(axis=0)).ravel())
ASTAT, AJOINT = {}, {}
short_a = dict(zip(LABELS_A, ['condition', 'relabeling 1', 'relabeling 2']))
for label in LABELS_A:
    df2 = int(den76[label]['df2'])
    alone = moderated_f(pd.Series(mod76[label]['stat|T_spatial'], index=U76), DF1, den76[label]['LR_spec'], df2, covariate=LMC76)
    np.testing.assert_allclose(alone.F.to_numpy(), ST76[short_a[label]].F.to_numpy(), rtol=1e-6)
    num = pd.Series(np.concatenate([mod76[label]['stat|T_spatial'], AM[label]['stat|T_spatial']]),
                    index=[f'g|{g}' for g in U76] + [f'n|{g}' for g in NEW])
    joint = moderated_f(num, DF1, np.concatenate([den76[label]['LR_spec'], AD[label]['LR_spec']]), df2,
                        covariate=np.concatenate([LMC76, LMCN]))
    joint['q_joint'] = false_discovery_control(joint.p.to_numpy())
    AJOINT[label] = joint
    a = joint.loc[[f'n|{g}' for g in NEW]].copy()
    a.index = NEW
    a['q'] = false_discovery_control(a.p.to_numpy())
    ASTAT[label] = a
A_CAL = float(np.mean(np.concatenate([ASTAT[l].p.to_numpy() for l in LABELS_A[1:]]) <= ALPHA))
print(f'AKI new-gene calibration share {A_CAL:.3f} (claims need ≤ {CALIBRATION_MAX})')
one76 = INV['mouse'].class_support3.reindex(U76).eq('1:1').to_numpy()
newcls = INV['mouse'].class_support3.reindex(NEW).fillna('absent')
called_new = AJOINT['species'].loc[[f'n|{g}' for g in NEW], 'q_joint'].to_numpy() <= ALPHA
called_76 = AJOINT['species'].loc[[f'g|{g}' for g in U76], 'q_joint'].to_numpy() <= ALPHA
non_new = newcls.isin(NON).to_numpy()
both_a = pd.DataFrame({'called': np.concatenate([called_76[one76], called_new[non_new]]),
                       'lmc': np.concatenate([LMC76[one76], LMCN[non_new]]),
                       'non': np.r_[np.zeros(one76.sum(), bool), np.ones(non_new.sum(), bool)]})
mh11 = pf.mantel_haenszel(both_a.non.to_numpy(), both_a.called.to_numpy(),
                          pd.qcut(both_a.lmc.rank(method='first'), 10, labels=False).to_numpy())
p11a = 2 * norm.sf(abs(mh11['z']))
primary('H11a AKI: positional calls, non-1:1 vs 1:1 (MH OR, two-sided)', p11a, f"OR {mh11['or_mh']:.2f}")
print(f"H11a: non-1:1 called {int(called_new[non_new].sum())}/{int(non_new.sum())}, 1:1 {int(called_76[one76].sum())}/"
      f"{int(one76.sum())}; OR {mh11['or_mh']:.2f}, two-sided p {p11a:.3g}")

# P-bar: 76's centred level, for U76 (its cached models) and the new genes (centred by U76's m(x))
PAIRS_A = [(i, c) for i in IRS for c in CONTROLS]


def pbar_tables():
    out = {}
    ok = mod76['species']['stat|converged'].astype(bool) & ~mod76['species']['stat|separated'].astype(bool)
    for label in LABELS_A:
        okl = mod76[label]['stat|converged'].astype(bool) & ~mod76[label]['stat|separated'].astype(bool)
        c, mx = centre_rows(mod76[label]['delta'], okl)
        out[short_a[label]] = (c, AM[label]['delta'] - mx)
    for i, k in PAIRS_A:
        okp = (ok & mod76['species'][f'converged|{i}'].astype(bool) & mod76['species'][f'converged|{k}'].astype(bool)
               & ~mod76['species'][f'separated|{i}'].astype(bool) & ~mod76['species'][f'separated|{k}'].astype(bool))
        c, mx = centre_rows(mod76['species'][f'curve|{i}'] - mod76['species'][f'curve|{k}'], okp)
        out[f'{i}−{k}'] = (c, AM['species'][f'curve|{i}'] - AM['species'][f'curve|{k}'] - mx)
    old = pd.DataFrame({k: np.nanmean(v[0], axis=1) / np.log(2) for k, v in out.items()}, index=U76)
    new = pd.DataFrame({k: np.nanmean(v[1], axis=1) / np.log(2) for k, v in out.items()}, index=NEW)
    return old, new, out


PB_OLD, PB_NEW, CENTRED_A = pbar_tables()
pair_cols = [f'{i}−{k}' for i, k in PAIRS_A]


def usable_mask(model):
    ok = model['stat|converged'].astype(bool) & ~model['stat|separated'].astype(bool)
    for n in CONTROLS + IRS:
        ok &= model[f'converged|{n}'].astype(bool) & ~model[f'separated|{n}'].astype(bool)
    return ok


USE_OLD, USE_NEW = usable_mask(mod76['species']), usable_mask(AM['species'])
print(f'Usable for P̄ (76\'s rule, every specimen fit too): U76 {int(USE_OLD.sum()):,}/{len(U76):,}, '
      f'new {int(USE_NEW.sum())}/{len(NEW)}')


def loss_flags(t):
    rep = (t.condition <= -EFFECT) & (t[pair_cols] < 0).all(axis=1)
    spec = (t['relabeling 1'].abs() < RATIO * t.condition.abs()) & (t['relabeling 2'].abs() < RATIO * t.condition.abs())
    return rep, spec


# PT enrichment in the control mice (76's rule, on the QC units)
POSITIONS_A = pd.read_csv(PKG / 'aki' / 'all_units_positions.csv', index_col=0)
with h5py.File(QC_AKI, 'r') as f:
    QC = read_elem(f['layers/counts']).tocsr()
    qc_index = pd.Index(read_elem(f['obs'][f['obs'].attrs['_index']]).astype(str))
    qc_var = pd.Index(read_elem(f['var'][f['var'].attrs['_index']]).astype(str))
assert qc_index.equals(POSITIONS_A.index) and qc_var.equals(UA.var_names)
lib_cols = qc_var.isin(MOUSE_ORTHO) & ~qc_var.isin(MT)
qc_lib = np.asarray(QC[:, lib_cols].sum(axis=1)).ravel()
on_curve = POSITIONS_A.pseudospace.notna().to_numpy()
is_pt = POSITIONS_A.label.isin(['PT-S1', 'PT-S2', 'PT-S3']).to_numpy()
genes_e = U76 + NEW
cols_e = qc_var.get_indexer(genes_e)
enrich = {}
for s in CONTROLS:
    ins = on_curve & POSITIONS_A['sample'].eq(s).to_numpy()
    a_, b_ = ins & is_pt, ins & ~is_pt
    with np.errstate(divide='ignore', invalid='ignore'):
        enrich[s] = np.log2((np.asarray(QC[a_][:, cols_e].sum(axis=0)).ravel() / qc_lib[a_].sum())
                            / (np.asarray(QC[b_][:, cols_e].sum(axis=0)).ravel() / qc_lib[b_].sum()))
ENRICH = pd.DataFrame(enrich, index=genes_e)
PT_ENRICHED = ENRICH.ge(PT_ENRICH).all(axis=1)
rep_old, spec_old = loss_flags(PB_OLD)
rep_new, spec_new = loss_flags(PB_NEW)
g_old = PT_ENRICHED.reindex(U76).to_numpy() & one76 & USE_OLD
g_new = PT_ENRICHED.reindex(NEW).to_numpy() & non_new & USE_NEW
tab11 = np.array([[int(rep_new[g_new].sum()), int((~rep_new[g_new]).sum())],
                  [int(rep_old[g_old].sum()), int((~rep_old[g_old]).sum())]])
h11b = fisher_exact(tab11, alternative='greater')
primary('H11b AKI: replicated loss among PT-enriched genes, non-1:1 > 1:1 (Fisher)', h11b.pvalue, f'OR {h11b.statistic:.2f}')
print(f'H11b: replicated loss among PT-enriched genes: non-1:1 {tab11[0, 0]}/{tab11[0].sum()}, 1:1 {tab11[1, 0]}/{tab11[1].sum()}; '
      f'OR {h11b.statistic:.2f}, one-sided p {h11b.pvalue:.3g}. Specific as well: non-1:1 {int((rep_new & spec_new)[g_new].sum())}, '
      f'1:1 {int((rep_old & spec_old)[g_old].sum())}')
SU = pd.read_csv(T76 / 'tables' / 'segment_units.csv')
np.testing.assert_allclose(SU['grid position'].to_numpy(), mod76['species']['grid'], atol=1e-12)
S_UNITS = SU['segment units'].to_numpy()


def loss_place(centred_rows):
    return S_UNITS[np.nanargmin(centred_rows, axis=1)]


place_old = loss_place(CENTRED_A['condition'][0][np.flatnonzero(rep_old & g_old)])
place_new = loss_place(CENTRED_A['condition'][1][np.flatnonzero(rep_new & g_new)])
tab11c = np.array([[int((place_new >= 2).sum()), int((place_new < 2).sum())],
                   [int((place_old >= 2).sum()), int((place_old < 2).sum())]])
h11c = fisher_exact(tab11c)
primary('H11c AKI: maximal loss in S3, non-1:1 vs 1:1 (Fisher, two-sided)', h11c.pvalue, f'OR {h11c.statistic:.2f}')
print(f'H11c: maximal loss in S3: non-1:1 {tab11c[0, 0]}/{tab11c[0].sum()}, 1:1 {tab11c[1, 0]}/{tab11c[1].sum()}; p {h11c.pvalue:.3g}')
H11 = PB_NEW.assign(usable=USE_NEW, cls=newcls.to_numpy(), q=ASTAT['species'].q.to_numpy(), q_joint=AJOINT['species'].loc[[f'n|{g}' for g in NEW], 'q_joint'].to_numpy(),
                    pt_enriched=PT_ENRICHED.reindex(NEW).to_numpy(), replicated_loss=rep_new.to_numpy(), specific=spec_new.to_numpy(),
                    loss_place=np.where(rep_new.to_numpy(), np.nan, np.nan))
H11.loc[(rep_new & g_new).to_numpy(), 'loss_place'] = place_new
H11.to_csv(tables / 'h11_aki_new_genes.csv')
pd.DataFrame({'H11b': tab11.ravel(), 'H11c': tab11c.ravel()},
             index=['non-1:1 yes', 'non-1:1 no', '1:1 yes', '1:1 no']).to_csv(tables / 'h11_tables.csv')
display(H11[H11.pt_enriched & H11.cls.isin(NON) & H11.usable].sort_values('condition').head(40).round(3))

LIT_AKI = pd.read_csv(project / 'docs' / 'results' / 'non-one-to-one' / 'literature_aki.csv')
rows11d = []
for _, c in LIT_AKI.iterrows():
    g = c.gene
    if g in PB_NEW.index:
        t, src, q_, use = PB_NEW.loc[g], 'outside the pool', ASTAT['species'].q.get(g), bool(USE_NEW[NEW.index(g)])
    elif g in PB_OLD.index:
        t, src, q_, use = PB_OLD.loc[g], 'U76', ST76['condition'].q.get(g), bool(USE_OLD[U76.index(g)])
    else:
        rows11d.append({**c.to_dict(), 'measured': 'no', 'verdict': 'not measurable'})
        continue
    if not use:
        rows11d.append({**c.to_dict(), 'measured': src, 'P̄ condition': t.condition, 'q': q_,
                        'verdict': 'not measurable (a fit hit zero counts)'})
        continue
    sign = 1 if c.expected == 'up' else -1
    agree = bool((np.sign(t[pair_cols]) == sign).all())
    ok = sign * t.condition >= EFFECT and agree
    rows11d.append({**c.to_dict(), 'measured': src, 'P̄ condition': t.condition, 'pairs agree': agree,
                    'P̄ pairs': ', '.join(f'{v:.2f}' for v in t[pair_cols]), 'q': q_,
                    'verdict': 'consistent' if ok else 'not consistent'})
H11D = pd.DataFrame(rows11d)
display(H11D.drop(columns=['note']).round(3))
H11D.to_csv(tables / 'h11d_named_aki_genes.csv', index=False)

## 15 · H12 · Named claims from the literature (section L of the protocol)

Each claim is read on the curve as stated: the mouse peak segment in both mice (each mouse's own centred curve),
or the stated order of paralogs; human claims from the label view. Verdicts: consistent, not consistent, not
measurable.

In [ ]:
LIT = pd.read_csv(project / 'docs' / 'results' / 'non-one-to-one' / 'literature_claims.csv')


def mouse_segment(units):
    return SEGMENTS[int(np.clip(np.floor(units), 0, 2))]


def read_gene(species, gene):
    C = CALLS[species]
    if gene not in PANEL[species]:
        return {'status': 'not on the panel'}
    if gene not in C.index:
        return {'status': 'not PT-expressed'}
    r = C.loc[gene]
    if not r.probe_clean:
        return {'status': 'probe-ambiguous'}
    out = {'status': 'measured', 'zonated': bool(r.zonated), 'Z': float(r.Z), 'label peak': r.label_peak}
    if species == 'mouse':
        sp = specimen_peaks('mouse', [C.index.get_loc(gene)])
        out.update({'peak (mouse units)': float(r.peak), 'peak segment': r.peak_segment,
                    'peak segments per mouse': '/'.join(mouse_segment(v[0]) for v in sp.values())})
    return out


def s1_cp10k(gene):
    d = W['mouse']['data']
    col = PANEL['mouse'].get_loc(gene)
    out = []
    for n in MOUSE:
        mk = (d['specimen'] == n) & (d['segment'] == 'PT-S1')
        out.append(float(d['counts'][mk][:, col].sum()) / d['library'][mk].sum() * 1e4)
    return out


rows12 = []
for _, c in LIT.iterrows():
    genes = [g.strip() for g in str(c.genes).split(';') if g.strip()]
    reads = {g: read_gene(c.species, g) for g in genes}
    measured = {g: v for g, v in reads.items() if v['status'] == 'measured'}
    expected = str(c.expected).split('|')
    verdict = 'not measurable'
    if c.kind == 'peak' and measured:
        g, v = next(iter(measured.items()))
        if c.species == 'mouse':
            per = v['peak segments per mouse'].split('/')
            verdict = ('not zonated' if not v['zonated'] else
                       'consistent' if all(p in expected for p in per) else 'not consistent')
        else:
            verdict = 'consistent' if v['label peak'] in expected else 'not consistent'
    elif c.kind == 'order' and len(measured) == len(genes) and c.species == 'mouse':
        ok_pairs = [pair_separated('mouse', a, b)[0] for a, b in zip(genes[:-1], genes[1:])]
        verdict = 'consistent' if all(ok_pairs) else 'not consistent'
    elif c.kind == 'present':
        g = genes[0]
        if reads[g]['status'] == 'not on the panel':
            verdict = 'not measurable'
        else:
            present = reads[g]['status'] in ('measured', 'probe-ambiguous')
            verdict = 'consistent' if present == (str(c.expected) == 'yes') else 'not consistent'
    elif c.kind == 'flat' and measured:
        verdict = 'consistent' if not next(iter(measured.values()))['zonated'] else 'not consistent'
    elif c.kind == 'shoulder' and measured:
        g, v = next(iter(measured.items()))
        s1 = s1_cp10k(g)
        reads[g]['S1 CP10k per mouse'] = s1
        verdict = 'consistent' if min(s1) >= 1 and v['peak segment'] == 'S3' else 'not consistent'
    rows12.append({**c.to_dict(), 'reads': json.dumps(reads, default=str), 'verdict': verdict})
H12 = pd.DataFrame(rows12)
display(H12[['id', 'species', 'genes', 'kind', 'expected', 'verdict']])
print(H12.verdict.value_counts())
H12.to_csv(tables / 'h12_literature_claims.csv', index=False)

# Thin-limb check (literature Q16): thin-limb genes on U_PT and on U_all
W_ALL = w_run('mouse', 'all', 'shared')
CALLS_ALL = annotate('mouse', W_ALL['calls'], W_ALL['data'])
TL = []
for g in ('Aox3', 'Sult1a1', 'Gstm2', 'Fst', 'Gdf15'):
    row = {'gene': g}
    for tag, c in (('U_PT', CALLS['mouse']), ('U_all', CALLS_ALL)):
        if g in c.index:
            row.update({f'{tag} detection': float(min(c.loc[g, f'detection_{n}'] for n in MOUSE)),
                        f'{tag} zonated': bool(c.loc[g, 'zonated']), f'{tag} peak': float(c.loc[g, 'peak'])})
        else:
            row[f'{tag} detection'] = 'not PT-expressed'
    TL.append(row)
THIN_CHECK = pd.DataFrame(TL)
display(THIN_CHECK.round(3))
THIN_CHECK.to_csv(tables / 'h12_thin_limb_check.csv', index=False)

## 16 · Multiplicity and the readout against the "adds nothing" rules

In [ ]:
PRIM = pd.DataFrame(PRIMARY)
PRIM['BH over the primary list'] = false_discovery_control(PRIM.p.fillna(1).to_numpy())
display(PRIM.round(4))
PRIM.to_csv(tables / 'primary_tests.csv', index=False)
h4m = H4.set_index('species').get('reading', pd.Series(dtype=str))
READOUT = pd.Series({
    'H0 positive control passes (mouse)': H0_PASS,
    'H1 holds (mouse OR ≥ 1.5, p < 0.05)': H1_HOLDS,
    'H7 differs from 1:1 genes (p < 0.05)': bool(p7 < ALPHA),
    'H7b: 71 calls that change with the family unit': int((H7B['gene called'] != H7B['family called']).sum()) if len(H7B) else 0,
    'H4 reading (mouse / human)': ' / '.join(str(h4m.get(s, 'not run')) for s in SPECIES),
    'H5 holds (mouse)': H5_HOLDS,
    'H11a differs (p < 0.05)': bool(p11a < ALPHA),
    'H11b holds (p < 0.05)': bool(h11b.pvalue < ALPHA)})
display(READOUT.to_frame('value'))

## 17 · Figures (plots only)

- **86a:** strong-zonation share by class and species; peak-position densities of strong genes by class (mouse
  units); male-biased share by class.
- **86b:** families on the curve: mouse members above human members, centred log2 curves on the shared curve
  position (ticks at the mouse cuts). Families: those with a strongly zonated probe-clean member and ≥ 2
  PT-expressed members in one species, the 8 with the largest member Z.
- **86c:** paralog co-positioning against the matched null; identity against similarity; tandem against dispersed.
- **86d:** family units across species: share called against matched 1:1 genes under the condition and the
  relabelings; the four called families with the smallest p, mouse and human family curves.
- **86e:** strongly zonated genes whose microdissection peak moves with sex: our curves beside Chen 2023 male and
  female segment means.
- **86f:** AKI: P̄ of PT-enriched genes by class; centred Δ of the eight PT-enriched non-1:1 genes with the largest
  replicated loss.
- **86g:** section L's named mouse genes on the curve, both mice.

In [ ]:
from matplotlib.colors import TwoSlopeNorm

CLASS_ORDER = ['1:1', '1:many', 'many:1', 'many:many', 'no_consensus', 'no_ortholog']
CUTS_X = list(MOUSE_CUTS)


def boundaries(ax, units=True, color=LIGHT):
    for b in ((1, 2) if units else CUTS_X):
        ax.axvline(b, color=color, lw=.6, zorder=0)


def wilson(k, n, z=1.96):
    k, n = np.asarray(k, float), np.maximum(np.asarray(n, float), 1)
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return np.vstack([p - (centre - half), (centre + half) - p])


def fig_a():
    fig, axes = plt.subplots(1, 3, figsize=(170 * MM, 45 * MM), gridspec_kw={'wspace': .4, 'width_ratios': [1.4, 1.2, 1]})
    ax = axes[0]
    width = .38
    for k, s in enumerate(SPECIES):
        t = H1_CLASSES.loc[s].reindex(CLASS_ORDER)
        x = np.arange(len(CLASS_ORDER)) + (k - .5) * width
        share = t['strong share'].to_numpy(float)
        err = wilson(t.strong.to_numpy(float), t.expressed.to_numpy(float))
        ax.bar(x, share, width, color=['#0072B2', '#D55E00'][k], yerr=err, error_kw={'lw': .5}, lw=0)
    ax.set_xticks(range(len(CLASS_ORDER)))
    ax.set_xticklabels(CLASS_ORDER, rotation=40, ha='right')
    ax.set_ylabel('strongly zonated share')
    ax = axes[1]
    m_ = CALLS['mouse']
    sm_ = m_[m_.strong & m_.probe_clean.astype(bool)]
    bins = np.linspace(0, 3, 25)
    for cls, color in (('1:1', CLASS_COLOR['1:1']), ('non-1:1', CLASS_COLOR['non-1:1'])):
        ax.hist(sm_.peak[sm_.cls.eq(cls)], bins=bins, density=True, histtype='step', color=color, lw=1.2)
    boundaries(ax)
    ax.set_xlabel('peak (mouse segment units)')
    ax.set_ylabel('density')
    ax = axes[2]
    sx = m_[m_.strong & m_.probe_clean.astype(bool) & m_.xiong_measured & m_.cls.isin(['1:1', 'non-1:1'])]
    share = sx.groupby('cls').male_biased.mean().reindex(['1:1', 'non-1:1'])
    n = sx.groupby('cls').size().reindex(['1:1', 'non-1:1'])
    k_ = sx.groupby('cls').male_biased.sum().reindex(['1:1', 'non-1:1'])
    ax.bar([0, 1], share, color=[CLASS_COLOR['1:1'], CLASS_COLOR['non-1:1']],
           yerr=wilson(k_.to_numpy(float), n.to_numpy(float)), error_kw={'lw': .5}, lw=0)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['1:1', 'non-1:1'])
    ax.set_ylabel('male-biased share')
    save(fig, 'fig86a_inventory')


def pick_families(n=8):
    rows = []
    for s in SPECIES:
        C = CALLS[s]
        for comp, g in C[C.component.ge(0) & C.non].groupby('component'):
            if len(g) >= 2 and (g.strong & g.probe_clean.astype(bool)).any():
                rows.append((float(g.Z.max()), int(comp)))
    return [c for _, c in sorted(set(rows), reverse=True)][:n]


def fig_b():
    fams = pick_families()
    blocks = []
    for comp in fams:
        for s in SPECIES:
            C = CALLS[s]
            g = C[C.component.eq(comp)].sort_values('peak')
            for gene in g.index:
                rows_ = C.index.get_loc(gene)
                curve = np.nanmean([PER_SPEC[s][n][rows_] for n in SPECIES[s]], axis=0)
                blocks.append((comp, s, gene, curve))
    if not blocks:
        return
    fig, ax = plt.subplots(figsize=(110 * MM, (3.0 * len(blocks) + 12) * MM))
    M = np.array([b[3] for b in blocks])
    ax.imshow(M, cmap='RdBu_r', norm=TwoSlopeNorm(0, -3, 3), aspect='auto', extent=(GRID[0], GRID[-1], len(blocks) - .5, -.5),
              interpolation='nearest')
    ax.set_yticks(range(len(blocks)))
    ax.set_yticklabels([b[2] for b in blocks])
    for k, b in enumerate(blocks):
        ax.get_yticklabels()[k].set_color('#0072B2' if b[1] == 'mouse' else '#D55E00')
    edges = [k + .5 for k in range(len(blocks) - 1) if blocks[k][0] != blocks[k + 1][0]]
    for y in edges:
        ax.axhline(y, color=INK, lw=.8)
    for x in CUTS_X:
        ax.axvline(x, color=INK, lw=.5, ls=':')
    ax.set_xlabel('elastic curve position')
    save(fig, 'fig86b_families')


def fig_c():
    fig, axes = plt.subplots(1, 4, figsize=(180 * MM, 45 * MM), gridspec_kw={'wspace': .45})
    s = 'mouse'
    if s in H4_PAIRS:
        t = H4_PAIRS[s]
        C = CALLS[s]
        ax = axes[0]
        ax.scatter(t['peak a'], t['peak b'], s=6, color=CLASS_COLOR['non-1:1'], lw=0)
        ax.plot([0, 3], [0, 3], color=LIGHT, lw=.6)
        for b in (1, 2):
            ax.axvline(b, color=LIGHT, lw=.5)
            ax.axhline(b, color=LIGHT, lw=.5)
        ax.set_xlabel('paralog a peak (mouse units)')
        ax.set_ylabel('paralog b peak')
        ax = axes[1]
        ax.hist(t['percentile separation'], bins=np.linspace(0, 1, 11), color=GREY, lw=0)
        ax.axvline(.5, color=INK, lw=.6, ls=':')
        ax.set_xlabel('separation percentile vs matched pairs')
        ax.set_ylabel('pairs')
    ax = axes[2]
    t = H5_PAIRS['mouse']
    ax.scatter(t.identity_mean, t.r, s=3, color=np.where(t.tandem, CLASS_COLOR['non-1:1'], GREY), lw=0, alpha=.7)
    bins = np.quantile(t.identity_mean, np.linspace(0, 1, 9))
    mid = pd.cut(t.identity_mean, bins, include_lowest=True)
    med = t.groupby(mid, observed=True).r.median()
    ax.plot([iv.mid for iv in med.index], med.to_numpy(), color=INK, lw=1)
    ax.axhline(H5.loc[(H5.species == 'mouse') & H5.setting.str.startswith('floor'), 'median r'].iloc[0], color=INK, lw=.6, ls=':')
    ax.set_xlabel('protein identity (%)')
    ax.set_ylabel('profile correlation r')
    ax = axes[3]
    nbm = pd.read_csv(tables / 'h6b_neighbour_pairs_mouse.csv')
    groups = [t.r[t.tandem].to_numpy(), t.r[~t.tandem].to_numpy(), nbm.r.to_numpy()]
    parts = ax.violinplot(groups, showmedians=True, widths=.8)
    for body, color in zip(parts['bodies'], [CLASS_COLOR['non-1:1'], GREY, LIGHT]):
        body.set_facecolor(color)
        body.set_alpha(.8)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['tandem paralogs', 'dispersed paralogs', 'neighbours'], rotation=30, ha='right')
    ax.set_ylabel('profile correlation r')
    save(fig, 'fig86c_paralogs')


def fig_d():
    fig = plt.figure(figsize=(170 * MM, 50 * MM))
    gs = fig.add_gridspec(1, 5, width_ratios=[1.3, 1, 1, 1, 1], wspace=.45)
    ax = fig.add_subplot(gs[0])
    shares = []
    for l in LABELS:
        qj = JOINT[l]
        fam = (qj.loc[[f'f|{k}' for k in FNAMES], 'q_joint'] <= ALPHA).mean()
        one = (qj.loc[[f'g|{g}' for g in np.array(GENES71)[one_to_one]], 'q_joint'] <= ALPHA).mean()
        shares.append((fam, one))
    shares = np.array(shares)
    x = np.arange(3)
    ax.bar(x - .2, shares[:, 0], .38, color=CLASS_COLOR['non-1:1'], lw=0)
    ax.bar(x + .2, shares[:, 1], .38, color=CLASS_COLOR['1:1'], lw=0)
    ax.set_xticks(x)
    ax.set_xticklabels(['species', 'relabel 1', 'relabel 2'], rotation=30, ha='right')
    ax.set_ylabel('share called')
    top = fam_table[fam_table.called].sort_values('p').head(4)
    for k, (_, r) in enumerate(top.iterrows()):
        ax = fig.add_subplot(gs[k + 1])
        j = FNAMES.index(r.family)
        ref, case = FM['species']['reference'][j], FM['species']['case'][j]
        ax.plot(GRID, (ref - ref.mean()) / np.log(2), color='#0072B2')
        ax.plot(GRID, (case - case.mean()) / np.log(2), color='#D55E00')
        for xc in CUTS_X:
            ax.axvline(xc, color=LIGHT, lw=.5)
        ax.set_title(r.label.split(' | ')[0][:22], loc='left')
        ax.set_xlabel('curve position')
        if k == 0:
            ax.set_ylabel('family log2, centred')
    save(fig, 'fig86d_family_units')


def fig_e():
    t = H3_MOVED[H3_MOVED.cls.eq('non-1:1')].sort_values('Z', ascending=False).head(4)
    if not len(t):
        return
    fig, axes = plt.subplots(2, len(t), figsize=(42 * MM * len(t), 62 * MM), gridspec_kw={'height_ratios': [2, 1], 'hspace': .5})
    axes = np.atleast_2d(axes).reshape(2, -1)
    for k, gene in enumerate(t.index):
        ax = axes[0, k]
        row = CALLS['mouse'].index.get_loc(gene)
        for n in MOUSE:
            ax.plot(GRID_UNITS['mouse'], PER_SPEC['mouse'][n][row], color=SPEC_COLOR[n])
        boundaries(ax)
        ax.set_title(gene, loc='left')
        ax.set_xlabel('mouse segment units')
        ax2 = axes[1, k]
        vals = c23.loc[gene, ['PTS1', 'PTS2', 'PTS3', 'fPTS1', 'fPTS2', 'fPTS3']].astype(float)
        ax2.bar(np.arange(3) - .2, np.log2(vals.iloc[:3] + 1), .38, color=INK, lw=0)
        ax2.bar(np.arange(3) + .2, np.log2(vals.iloc[3:] + 1), .38, color=GREY, lw=0)
        ax2.set_xticks(range(3))
        ax2.set_xticklabels(SEGMENTS)
        if k == 0:
            ax.set_ylabel('log2, centred')
            ax2.set_ylabel('log2 TPM')
    save(fig, 'fig86e_sex')


def fig_f():
    fig, axes = plt.subplots(1, 2, figsize=(150 * MM, 48 * MM), gridspec_kw={'wspace': .35, 'width_ratios': [1, 1.6]})
    ax = axes[0]
    vals = [PB_OLD.condition[g_old].to_numpy(), PB_NEW.condition[g_new].to_numpy()]
    vals = [np.clip(v, -8, 8) for v in vals]
    parts = ax.violinplot(vals, showmedians=True, widths=.8)
    for body, color in zip(parts['bodies'], [CLASS_COLOR['1:1'], CLASS_COLOR['non-1:1']]):
        body.set_facecolor(color)
        body.set_alpha(.7)
    ax.axhline(0, color=LIGHT, lw=.6)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['1:1', 'non-1:1'])
    ax.set_ylabel('P̄ (IR − control, log2)')
    ax = axes[1]
    sel = PB_NEW[g_new & rep_new].sort_values('condition').head(8).index
    cmap = plt.get_cmap('tab10')
    for k, gene in enumerate(sel):
        i = NEW.index(gene)
        ax.plot(S_UNITS, CENTRED_A['condition'][1][i] / np.log(2), color=cmap(k % 10), lw=.9, label=gene)
    for b in (1, 2):
        ax.axvline(b, color=LIGHT, lw=.6)
    ax.axhline(0, color=LIGHT, lw=.5)
    ax.legend(fontsize=5, ncol=2, loc='lower left')
    ax.set_xlabel('segment units (control cuts)')
    ax.set_ylabel('centred Δ (log2)')
    save(fig, 'fig86f_aki')


def fig_g():
    genes = []
    for _, c in H12[H12.species.eq('mouse')].iterrows():
        for g in str(c.genes).split(';'):
            g = g.strip()
            if g in CALLS['mouse'].index and g not in genes:
                genes.append(g)
    genes = genes[:16]
    if not genes:
        return
    ncol = 4
    nrow = int(np.ceil(len(genes) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(170 * MM, 32 * MM * nrow), sharex=True, gridspec_kw={'hspace': .55, 'wspace': .3})
    for ax, gene in zip(np.ravel(axes), genes):
        row = CALLS['mouse'].index.get_loc(gene)
        for n in MOUSE:
            ax.plot(GRID_UNITS['mouse'], PER_SPEC['mouse'][n][row], color=SPEC_COLOR[n])
        boundaries(ax)
        ax.axhline(0, color=LIGHT, lw=.4)
        ax.set_title(gene, loc='left')
    for ax in np.ravel(axes)[len(genes):]:
        ax.axis('off')
    save(fig, 'fig86g_literature_genes')


def fig_h():
    pairs = [('Slc5a2', 'Slc5a1'), ('Slc7a9', 'Slc3a1')]
    thin = [g for g in ('Aox3', 'Sult1a1', 'Gstm2', 'Fst', 'Gdf15') if g in CALLS_ALL.index]
    fig, axes = plt.subplots(2, max(len(pairs), len(thin)), figsize=(170 * MM, 62 * MM), sharex=True,
                             gridspec_kw={'hspace': .5, 'wspace': .3})
    for k, (a, b) in enumerate(pairs):
        ax = axes[0, k]
        for gene, ls in ((a, '-'), (b, '--')):
            if gene in CALLS['mouse'].index:
                row = CALLS['mouse'].index.get_loc(gene)
                for n in MOUSE:
                    ax.plot(GRID_UNITS['mouse'], PER_SPEC['mouse'][n][row], color=SPEC_COLOR[n], ls=ls)
        boundaries(ax)
        ax.set_title(f'{a} / {b}', loc='left')
    for ax in axes[0, len(pairs):]:
        ax.axis('off')
    zc = W_ALL['z']['centred']
    genes_all = list(CALLS_ALL.index)
    for k, gene in enumerate(thin):
        ax = axes[1, k]
        row = genes_all.index(gene)
        for n in MOUSE:
            ax.plot(GRID_UNITS['mouse'], zc[n][row], color=SPEC_COLOR[n])
            if gene in CALLS['mouse'].index:
                ax.plot(GRID_UNITS['mouse'], PER_SPEC['mouse'][n][CALLS['mouse'].index.get_loc(gene)], color=SPEC_COLOR[n], ls=':')
        boundaries(ax)
        ax.set_title(gene, loc='left')
        ax.set_xlabel('mouse segment units')
    for ax in axes[1, len(thin):]:
        ax.axis('off')
    save(fig, 'fig86h_controls')


for f in (fig_a, fig_b, fig_c, fig_d, fig_e, fig_f, fig_g, fig_h):
    f()

## 18 · Outputs and run record

In [ ]:
import platform
from importlib.metadata import version

record = {
    'notebook_logic_version': NOTEBOOK_LOGIC_VERSION,
    'protocol': 'docs/results/pt-non-one-to-one.md',
    'guards': GUARDS, 'external_sha256': EXTERNAL_SHA,
    'units': {s: int((U_PT & (SPECIMEN == s)).sum()) for s in MOUSE + HUMAN},
    'pt_expressed': {s: int(len(CALLS[s])) for s in SPECIES},
    'zonated': {s: int(CALLS[s].zonated.sum()) for s in SPECIES}, 'strong': {s: int(CALLS[s].strong.sum()) for s in SPECIES},
    'family_units': len(FNAMES), 'aki_new_genes': len(NEW), 'aki_new_calibration_share': A_CAL,
    'family_calibration_share': F_CAL, 'primary_tests': PRIM.to_dict(orient='records'),
    'readout': {k: (v.item() if hasattr(v, 'item') else v) for k, v in READOUT.items()},
    'stage_seconds': SECONDS, 'n_jobs': N_JOBS, 'seed': SEED,
    'versions': {'python': platform.python_version(),
                 **{m_: version(m_) for m_ in ('numpy', 'scipy', 'pandas', 'anndata', 'matplotlib')}},
    'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f"Wall time {record['wall_seconds']:.0f} s")